In [ ]:
# C00B - Checkout de Colab en un commit fijo y carga de snapshots no publicados
import os, re, subprocess
from pathlib import Path

IS_COLAB_BOOT = "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ


def _run_checked(args, *, cwd=None, action="comando"):
    """Ejecuta un comando y conserva el detalle del error para diagnosticarlo."""
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if result.returncode:
        detail = (result.stderr or result.stdout or "sin detalle del proceso").strip()
        raise RuntimeError(f"{action} fall? (c?digo {result.returncode}): {detail}")
    return result.stdout.strip()


if not IS_COLAB_BOOT:
    print("Entorno local: no se clona ni se instalan paquetes en esta celda.")
else:
    url = os.environ.get("ML2_REPO_URL") or input("URL HTTPS GitHub: ").strip()
    commit = os.environ.get("ML2_REPO_COMMIT") or input("SHA completo del commit: ").strip()
    if not url.startswith("https://") or not re.fullmatch(r"[0-9a-fA-F]{40,64}", commit):
        raise ValueError("Proporciona una URL HTTPS y un SHA completo de 40 o 64 caracteres.")

    repo = Path("/content/Proyecto_ML2_Final")
    if repo.exists():
        if not (repo / ".git").is_dir():
            raise RuntimeError(f"{repo} existe, pero no es un checkout Git. Ren?mbralo o elim?nalo en Colab y vuelve a ejecutar.")
        origin = _run_checked(
            ["git", "-C", str(repo), "remote", "get-url", "origin"],
            action="Consultar el origin del checkout",
        )
        if origin != url:
            raise RuntimeError(f"El origin existente ({origin}) no coincide con la URL solicitada ({url}).")
        _run_checked(
            ["git", "-C", str(repo), "fetch", "--depth", "1", "origin", commit],
            action="Descargar el commit solicitado",
        )
    else:
        _run_checked(
            ["git", "clone", "--filter=blob:none", "--no-checkout", url, str(repo)],
            action="Clonar el repositorio",
        )
        _run_checked(
            ["git", "-C", str(repo), "fetch", "--depth", "1", "origin", commit],
            action="Descargar el commit solicitado",
        )

    _run_checked(
        ["git", "-C", str(repo), "checkout", "--detach", commit],
        action="Fijar el checkout al commit solicitado",
    )
    actual = _run_checked(
        ["git", "-C", str(repo), "rev-parse", "HEAD"],
        action="Verificar el commit activo",
    )
    if actual.lower() != commit.lower():
        raise AssertionError(f"El checkout qued? en {actual}, no en el SHA solicitado {commit}.")

    os.chdir(repo)
    os.environ["ML2_PROJECT_DIR"] = str(repo)
    req = repo / "requirements-colab.txt"
    if not req.is_file():
        raise FileNotFoundError(f"Falta el archivo de dependencias: {req}")

    private_snapshots = [
        repo / "data/snapshots/mercado_yahoo_finance_20261112T055009553793Z.csv",
        repo / "data/snapshots/embi_colombia_bcrpdata_20261112T055012253929Z.csv",
    ]
    missing = [snapshot for snapshot in private_snapshots if not snapshot.is_file()]
    if missing:
        for target in missing:
            target.parent.mkdir(parents=True, exist_ok=True)
        from google.colab import files
        print("Sube los snapshots locales autorizados que Git no distribuye:")
        uploaded = files.upload()
        for target in missing:
            if target.name not in uploaded:
                raise FileNotFoundError(f"No se recibi? el snapshot requerido: {target.name}.")
            target.write_bytes(uploaded[target.name])

    print(f"Checkout fijado a {actual}; paquetes base no se reinstalan.")
    print("C03 verifica versiones y, si falta, instala Streamlit con el pin de requirements-colab.txt.")
    print("C02 valida la lista completa de datos y snapshots requeridos.")


# Proyecto Integrador de Machine Learning 2

## Predicción de eventos extremos en la TRM colombiana

**Fases:** 1, 2 y 3  
**Entorno:** Google Colab o ejecución local  
**Datos primarios:** snapshots validados y archivos oficiales de apoyo  
**Evaluación:** selección mediante validación temporal y evaluación final en test congelado

> La corrida del notebook la realiza el estudiante. Los resultados informados son los producidos por la ejecución registrada de este notebook.


## Política de datos, trazabilidad y evaluación

1. Los snapshots seleccionados en el manifiesto son las fuentes primarias de esta corrida. Las consultas actuales a las fuentes oficiales o comerciales son verificaciones opcionales y nunca sustituyen los snapshots.
2. Si falta un archivo primario, no pasa el chequeo de esquema o su hash no coincide, el notebook se detiene con un error visible. No se interpolan ni inventan datos.
3. La TRM procesada se reconstruye desde la TRM cruda y se compara con trm_aprobada_fase1_20261112T055009052854Z.csv, que es una referencia de comparación, no una fuente primaria.
4. Los conteos se informan por etapa: TRM válida, retornos, serie etiquetada GARCH y conjunto de clasificación con rezagos y objetivo.
5. El test final se conserva congelado. No se utiliza para elegir modelo, hiperparámetros ni umbral.
6. Los registros de versiones, manifiesto y verificaciones se guardan en la carpeta única de la corrida bajo results/runs/run_<id>/.

Este notebook reproduce la corrida validada del 12/11/2026 con los datos y hashes fijados; no está pensado para actualizarse con datos nuevos sin revisar los asserts.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import re
import sys

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ
MODE_NAME = "Google Colab" if IS_COLAB else "Local"

def is_project_root(path):
    path = Path(path)
    return (
        path.is_dir()
        and (path / "requirements.txt").is_file()
        and (path / "data" / "trm_spikes_dataset.csv").is_file()
        and (path / "data" / "tasa_intervencion_banrep_real.csv").is_file()
        and (path / "data" / "snapshots").is_dir()
    )

project_env = os.environ.get("ML2_PROJECT_DIR")
if project_env:
    PROJECT_DIR = Path(project_env).expanduser().resolve()
    if not is_project_root(PROJECT_DIR):
        raise FileNotFoundError(f"ML2_PROJECT_DIR no es raiz valida: {PROJECT_DIR}")
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    if IS_COLAB and Path("/content").is_dir():
        candidates.extend(p for p in Path("/content").iterdir() if p.is_dir())
    PROJECT_DIR = next((Path(p) for p in dict.fromkeys(candidates) if is_project_root(p)), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "No encuentro la raiz del repositorio con requirements.txt y data/. "
            "En Colab, clona el repositorio bajo /content y vuelve a ejecutar C02, "
            "o define ML2_PROJECT_DIR. En local, abre el notebook desde entrega/."
        )

DATA_DIR = PROJECT_DIR / "data"
SNAPSHOT_DIR = DATA_DIR / "snapshots"
required_relative = [
    Path("requirements.txt"),
    Path("data/trm_spikes_dataset.csv"),
    Path("data/tasa_intervencion_banrep_real.csv"),
    Path("data/snapshots/trm_datos_gov_co_20261112T054932091186Z.csv"),
    Path("data/snapshots/mercado_yahoo_finance_20261112T055009553793Z.csv"),
    Path("data/snapshots/embi_colombia_bcrpdata_20261112T055012253929Z.csv"),
    Path("data/snapshots/trm_aprobada_fase1_20261112T055009052854Z.csv"),
    Path("data/snapshots/trm_datos_gov_co_20261112T050017416922Z.csv"),
]
missing = [str(p) for p in required_relative if not (PROJECT_DIR / p).is_file()]
if missing:
    raise FileNotFoundError("Faltan archivos requeridos:\n- " + "\n- ".join(missing))

RESUME_RUN_ID = None
MODO_RAPIDO = False  # True reduce grillas/particiones; solo para diagnóstico.
if RESUME_RUN_ID is not None:
    if not re.fullmatch(r"\d{8}T\d{12}Z", RESUME_RUN_ID):
        raise ValueError("RESUME_RUN_ID debe tener formato YYYYMMDDTHHMMSSffffffZ.")
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = PROJECT_DIR / "results" / "runs" / f"run_{RUN_ID}"
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f"No existe la corrida: {RUN_DIR}")
else:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    RUN_DIR = PROJECT_DIR / "results" / "runs" / f"run_{RUN_ID}"
    RUN_DIR.mkdir(parents=True, exist_ok=False)

print(f"Entorno: {MODE_NAME}; repositorio: {PROJECT_DIR}")
print(f"Datos: {DATA_DIR}; snapshots: {SNAPSHOT_DIR}")
print(f"run_id UTC: {RUN_ID}; RUN_DIR: {RUN_DIR}")
print("Archivos requeridos presentes.")

In [ ]:
from pathlib import Path
import importlib
import importlib.metadata as metadata
import json
import platform
import re
import subprocess
import sys
import tempfile
import warnings

INSTALL_OPTIONAL = {
    "streamlit": True,
    "shap": False,
}

if "PROJECT_DIR" not in globals() or not Path(PROJECT_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: PROJECT_DIR no está definido.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no está definido.")
REQUIREMENTS_NAME = "requirements-colab.txt" if IS_COLAB else "requirements.txt"
REQUIREMENTS_PATH = Path(PROJECT_DIR) / REQUIREMENTS_NAME
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(f"Falta {REQUIREMENTS_PATH}")

def canon(name):
    return re.sub(r"[-_.]+", "-", name).lower()

requirements_text = REQUIREMENTS_PATH.read_text(encoding="utf-8")
pins = {}
for line in requirements_text.splitlines():
    match = re.match(r"^\s*([A-Za-z0-9_.-]+)\s*==\s*([^;\s#]+)", line)
    if match:
        pins[canon(match.group(1))] = match.group(2)
python_match = re.search(
    r"(?im)^\s*#.*\bPython\s+(\d+\.\d+(?:\.\d+)?)",
    requirements_text,
)
reference_python = python_match.group(1) if python_match else "no declarado"

PACKAGE_MODULES = {
    "arch": "arch",
    "ipython": "IPython",
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "pandas": "pandas",
    "requests": "requests",
    "scikit-learn": "sklearn",
    "statsmodels": "statsmodels",
    "yfinance": "yfinance",
}
CRITICAL = {
    "arch", "joblib", "matplotlib", "numpy", "pandas", "requests",
    "scikit-learn", "statsmodels", "yfinance",
}
PROTECTED = {"numpy", "pandas", "scikit-learn"}

def get_version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return None

missing_critical = [name for name in CRITICAL if get_version(name) is None]
if missing_critical:
    raise ImportError(
        "Faltan paquetes críticos: " + ", ".join(sorted(missing_critical))
        + ". Instálalos en este entorno y vuelve a ejecutar C03."
    )

for distribution, module in PACKAGE_MODULES.items():
    try:
        importlib.import_module(module)
    except Exception as exc:
        if distribution in CRITICAL:
            raise ImportError(
                f"El paquete crítico {distribution!r} no se pudo importar: {exc}"
            ) from exc
        warnings.warn(
            f"No se pudo importar {module!r}; se continúa sin esa función: {exc}",
            RuntimeWarning,
        )

def install_optional(distribution):
    if get_version(distribution) is not None:
        return "ya instalado"
    package_spec = distribution + (f"=={pins[canon(distribution)]}" if canon(distribution) in pins else "")
    try:
        with tempfile.TemporaryDirectory() as temp_dir:
            report_path = Path(temp_dir) / "pip_plan.json"
            dry_run = subprocess.run(
                [
                    sys.executable, "-m", "pip", "install", "--dry-run",
                    "--report", str(report_path), "--upgrade-strategy",
                    "only-if-needed", package_spec,
                ],
                capture_output=True,
                text=True,
            )
            if dry_run.returncode != 0:
                raise RuntimeError(dry_run.stderr[-2000:])
            plan = json.loads(report_path.read_text(encoding="utf-8"))
            planned = {
                canon(item["metadata"]["name"])
                for item in plan.get("install", [])
            }
            conflicts = planned & PROTECTED
            if conflicts:
                return (
                    "no instalado: el plan modificaría paquetes protegidos "
                    + ", ".join(sorted(conflicts))
                )
            result = subprocess.run(
                [
                    sys.executable, "-m", "pip", "install",
                    "--upgrade-strategy", "only-if-needed", package_spec,
                ],
                capture_output=True,
                text=True,
            )
            if result.returncode != 0:
                raise RuntimeError(result.stderr[-2000:])
        return f"instalado; versión {get_version(distribution)}"
    except Exception as exc:
        warnings.warn(
            f"No se pudo instalar {distribution}; el notebook continúa: {exc}",
            RuntimeWarning,
        )
        return f"falló la instalación: {type(exc).__name__}: {exc}"

optional_status = {}
for distribution, enabled in INSTALL_OPTIONAL.items():
    if enabled:
        optional_status[distribution] = install_optional(distribution)
    elif get_version(distribution):
        optional_status[distribution] = "ya instalado; instalación desactivada"
    else:
        optional_status[distribution] = "no instalado; instalación desactivada"

records = [{
    "paquete": "python",
    "version_efectiva": platform.python_version(),
    "version_en_requirements": reference_python,
    "estado_instalacion_opcional": "",
}]
for distribution in PACKAGE_MODULES:
    records.append({
        "paquete": distribution,
        "version_efectiva": get_version(distribution) or "no instalado",
        "version_en_requirements": pins.get(canon(distribution), "sin pin"),
        "estado_instalacion_opcional": "",
    })
for distribution, status in optional_status.items():
    records.append({
        "paquete": distribution,
        "version_efectiva": get_version(distribution) or "no instalado",
        "version_en_requirements": pins.get(canon(distribution), "sin pin"),
        "estado_instalacion_opcional": status,
    })

import pandas as pd

versiones_df = pd.DataFrame(records)
if reference_python == "no declarado":
    warnings.warn("Python no está declarado en requirements.txt.", RuntimeWarning)
elif platform.python_version() != reference_python:
    warnings.warn(
        f"Python efectivo {platform.python_version()} difiere de requirements.txt "
        f"({reference_python}).",
        RuntimeWarning,
    )
package_rows = versiones_df[versiones_df["paquete"].ne("python")]
mismatches = package_rows[
    package_rows["version_en_requirements"].ne("sin pin")
    & package_rows["version_efectiva"].ne(package_rows["version_en_requirements"])
]
if not mismatches.empty:
    warnings.warn(
        "Hay diferencias de versiones frente a requirements.txt. "
        "Se registran como advertencias; no se modifica ese archivo.",
        RuntimeWarning,
    )

print(versiones_df.to_string(index=False))
versiones_path = Path(RUN_DIR) / "registro_versiones.csv"
versiones_df.to_csv(versiones_path, index=False)
print(f"Registro guardado en: {versiones_path}")


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

if "PROJECT_DIR" not in globals() or not Path(PROJECT_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: PROJECT_DIR no está definido.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no está definido.")

PRIMARY_FILES = {
    "trm_raw": "trm_datos_gov_co_20261112T054932091186Z.csv",
    "market": "mercado_yahoo_finance_20261112T055009553793Z.csv",
    "embi": "embi_colombia_bcrpdata_20261112T055012253929Z.csv",
}
PROCESSED_REFERENCE = "trm_aprobada_fase1_20261112T055009052854Z.csv"
RATE_FILE = Path(DATA_DIR) / "tasa_intervencion_banrep_real.csv"
TRM_REFERENCE_FILE = Path(DATA_DIR) / "trm_spikes_dataset.csv"
EXPECTED_SHA256 = {
    "trm_datos_gov_co_20261112T054932091186Z.csv":
        "6bc5e59e0aaf33ff2adf641a1e4e7b56899b95b723d82c0dfa8fe0777bae5bc5",
    "mercado_yahoo_finance_20261112T055009553793Z.csv":
        "c35169a78331707f3f516aeac4791d1d62ef80c6570d4bc3ec4e88a7ad2a2499",
    "embi_colombia_bcrpdata_20261112T055012253929Z.csv":
        "2d07cb5024f8594c0681fc27108750ec2c412de18ce248e093fe1afae4eee880",
    PROCESSED_REFERENCE:
        "5a5816d4c9c1696081c7ffcaa4fc357a0f5db455bd2bbccb5c6a342089bfd0d5",
    "tasa_intervencion_banrep_real.csv":
        "e3eac132dc12881990224d759f14c2d5e4d986c579f4ad055cd37b19162fab7a",
    "trm_spikes_dataset.csv":
        "7370b366b089691f42bdf5ddc1ad7932e2c12b7dc4c5fe68925692a44a1388d0",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

required_paths = [
    Path(PROJECT_DIR) / "requirements.txt",
    TRM_REFERENCE_FILE,
    RATE_FILE,
    Path(SNAPSHOT_DIR),
] + [
    Path(SNAPSHOT_DIR) / name
    for name in (*PRIMARY_FILES.values(), PROCESSED_REFERENCE,
                 "trm_datos_gov_co_20261112T050017416922Z.csv")
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan archivos requeridos:\\n- " + "\\n- ".join(missing))

for filename, expected in EXPECTED_SHA256.items():
    if filename == RATE_FILE.name:
        path = RATE_FILE
    elif filename == TRM_REFERENCE_FILE.name:
        path = TRM_REFERENCE_FILE
    else:
        path = Path(SNAPSHOT_DIR) / filename
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(
            f"Hash SHA-256 inesperado para {path.name}. "
            f"Esperado: {expected}; obtenido: {actual}"
        )

trm_raw = pd.read_csv(Path(SNAPSHOT_DIR) / PRIMARY_FILES["trm_raw"])
if not {"fuente", "fecha_consulta_utc", "fecha", "trm"}.issubset(trm_raw.columns):
    raise ValueError("Snapshot TRM primario sin esquema o metadata requerida.")
trm_raw["fecha"] = pd.to_datetime(trm_raw["fecha"], errors="coerce").dt.normalize()
trm_raw["trm"] = pd.to_numeric(trm_raw["trm"], errors="coerce")
if trm_raw[["fecha", "trm"]].isna().any().any():
    raise ValueError("El snapshot TRM tiene fechas o valores faltantes.")
if trm_raw["fecha"].duplicated().any() or not trm_raw["fecha"].is_monotonic_increasing:
    raise ValueError("Las fechas TRM deben ser únicas y estar ordenadas.")
if len(trm_raw) != 8364 or (trm_raw["trm"] <= 0).any():
    raise ValueError(f"TRM válida esperada: 8.364 filas; halladas {len(trm_raw)}.")

market = pd.read_csv(Path(SNAPSHOT_DIR) / PRIMARY_FILES["market"])
market_required = {
    "fuente", "fecha_consulta_utc", "date", "brent_close", "wti_close",
    "dxy_close", "mxn_close", "clp_close", "pen_close",
}
if not market_required.issubset(market.columns):
    raise ValueError(f"Snapshot de mercado incompleto: {market_required - set(market.columns)}")
market["date"] = pd.to_datetime(market["date"], errors="coerce").dt.normalize()
if market["date"].isna().any() or market["date"].duplicated().any():
    raise ValueError("El snapshot de mercado tiene fechas inválidas o duplicadas.")

embi = pd.read_csv(Path(SNAPSHOT_DIR) / PRIMARY_FILES["embi"])
if not {"fuente", "fecha_consulta_utc", "date", "embi_colombia"}.issubset(embi.columns):
    raise ValueError("Snapshot EMBI primario sin esquema o metadata requerida.")
embi["date"] = pd.to_datetime(embi["date"], errors="coerce").dt.normalize()
if embi["date"].isna().any() or embi["date"].duplicated().any():
    raise ValueError("El snapshot EMBI tiene fechas inválidas o duplicadas.")

banrep = pd.read_csv(RATE_FILE)
if not {"fecha_efectiva", "tasa_intervencion"}.issubset(banrep.columns):
    raise ValueError("El CSV BanRep debe tener fecha_efectiva y tasa_intervencion.")
banrep["fecha_efectiva"] = pd.to_datetime(
    banrep["fecha_efectiva"], errors="coerce"
).dt.normalize()
banrep["tasa_intervencion"] = pd.to_numeric(
    banrep["tasa_intervencion"], errors="coerce"
)
if banrep[["fecha_efectiva", "tasa_intervencion"]].isna().any().any():
    raise ValueError("El CSV BanRep contiene fecha o tasa inválida.")
if banrep["fecha_efectiva"].duplicated().any():
    raise ValueError("El CSV BanRep tiene fechas efectivas duplicadas.")
if not banrep["fecha_efectiva"].is_monotonic_increasing:
    raise ValueError("El CSV BanRep no está ordenado por fecha efectiva.")
if not banrep["tasa_intervencion"].between(0, 100).all():
    raise ValueError("La tasa BanRep debe expresarse en puntos porcentuales.")

trm_reference = pd.read_csv(TRM_REFERENCE_FILE)
required_trm_cols = {"fecha", "trm", "log_return", "spike", "target_next_day"}
if not required_trm_cols.issubset(trm_reference.columns):
    raise ValueError(f"CSV TRM de comparación incompleto: {required_trm_cols - set(trm_reference.columns)}")
if len(trm_reference) != 8354:
    raise ValueError(f"Se esperaban 8.354 filas; se encontraron {len(trm_reference)}.")

print("Archivos, esquemas y hashes validados.")
print("BanRep y trm_spikes_dataset.csv no incluyen metadata de consulta embebida.")


In [ ]:
from pathlib import Path
import hashlib
import warnings
import pandas as pd

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no está definido.")
if "SNAPSHOT_DIR" not in globals() or not Path(SNAPSHOT_DIR).is_dir():
    raise RuntimeError("No existe la carpeta snapshots.")
if "EXPECTED_SHA256" not in globals():
    raise RuntimeError("Ejecuta C04 primero para definir los hashes.")

PRIMARY_SNAPSHOT_NAMES = {
    "trm_datos_gov_co_20261112T054932091186Z.csv",
    "mercado_yahoo_finance_20261112T055009553793Z.csv",
    "embi_colombia_bcrpdata_20261112T055012253929Z.csv",
}
ROLES = {
    "trm_datos_gov_co_20261112T054932091186Z.csv": "TRM cruda primaria",
    "mercado_yahoo_finance_20261112T055009553793Z.csv": "Mercado primario",
    "embi_colombia_bcrpdata_20261112T055012253929Z.csv": "EMBI primario",
    "trm_aprobada_fase1_20261112T055009052854Z.csv":
        "TRM procesada de referencia; no primaria",
}
EXPECTED_HISTORICAL_HASH = (
    "07f44229e414950dadd06a793a5a05204db474b07acb3518421784be45b7d8dc"
)

def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def manifest_row(path, role, is_primary_snapshot=False):
    frame = pd.read_csv(path)
    date_col = next(
        (col for col in ("date", "fecha", "fecha_efectiva") if col in frame.columns),
        None,
    )
    if date_col is None:
        raise ValueError(f"No se encontró columna de fecha en {path.name}.")
    dates = pd.to_datetime(frame[date_col], errors="coerce").dropna()
    if dates.empty:
        raise ValueError(f"No hay fechas válidas en {path.name}.")

    has_source = "fuente" in frame.columns and frame["fuente"].notna().any()
    if has_source:
        source = " | ".join(frame["fuente"].dropna().astype(str).unique())
    else:
        source = "no incluida en archivo"
        message = f"{path.name}: falta metadata fuente."
        if is_primary_snapshot:
            raise ValueError(message)
        warnings.warn(message, RuntimeWarning)

    has_capture = (
        "fecha_consulta_utc" in frame.columns
        and frame["fecha_consulta_utc"].notna().any()
    )
    if has_capture:
        captures = frame["fecha_consulta_utc"].dropna().astype(str).unique()
        if len(captures) != 1:
            raise ValueError(f"{path.name}: fecha_consulta_utc no es única.")
        capture = captures[0]
    else:
        capture = "no incluida en archivo"
        message = f"{path.name}: falta metadata fecha_consulta_utc."
        if is_primary_snapshot:
            raise ValueError(message)
        warnings.warn(message, RuntimeWarning)

    digest = file_hash(path)
    expected = EXPECTED_SHA256.get(path.name)
    hash_status = "sin hash esperado fijado"
    if path.name == "trm_datos_gov_co_20261112T050017416922Z.csv":
        expected = EXPECTED_HISTORICAL_HASH
        if digest == expected:
            hash_status = "coincide"
        else:
            hash_status = "difiere; advertencia"
            warnings.warn(
                f"Hash diferente para {path.name}: esperado {expected}, obtenido {digest}. "
                "Se registra la diferencia y se continúa.",
                RuntimeWarning,
            )
    elif expected is not None:
        if digest != expected:
            raise RuntimeError(f"Hash inesperado para archivo fijado {path.name}: {digest}")
        hash_status = "coincide"

    return {
        "archivo": str(path.relative_to(PROJECT_DIR)),
        "fuente": source,
        "filas": len(frame),
        "fecha_inicial": dates.min().date().isoformat(),
        "fecha_final": dates.max().date().isoformat(),
        "sha256": digest,
        "sha256_esperado": expected or "no fijado",
        "estado_hash": hash_status,
        "fecha_captura_utc": capture,
        "rol": role,
    }

snapshot_files = sorted(Path(SNAPSHOT_DIR).glob("*.csv"))
if not snapshot_files:
    raise FileNotFoundError(f"No hay snapshots CSV en {SNAPSHOT_DIR}.")
rows = []
for path in snapshot_files:
    rows.append(manifest_row(
        path,
        ROLES.get(path.name, "Snapshot disponible; uso no acreditado"),
        is_primary_snapshot=path.name in PRIMARY_SNAPSHOT_NAMES,
    ))

support_files = [
    (Path(DATA_DIR) / "tasa_intervencion_banrep_real.csv",
     "CSV oficial BanRep; insumo primario de apoyo"),
    (Path(DATA_DIR) / "trm_spikes_dataset.csv",
     "Conjunto de comparación; no fuente primaria"),
]
for path, role in support_files:
    if not path.is_file():
        raise FileNotFoundError(f"Falta el archivo de apoyo {path}.")
    rows.append(manifest_row(path, role, is_primary_snapshot=False))

manifest_df = pd.DataFrame(rows)
manifest_path = Path(RUN_DIR) / "manifiesto_fuentes.csv"
manifest_df.to_csv(manifest_path, index=False)
print(manifest_df.to_string(index=False))
print(f"Manifiesto guardado en: {manifest_path}")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import re
import requests
import pandas as pd
import yfinance as yf

if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no está definido.")
if "trm_raw" not in globals() or "market" not in globals() or "embi" not in globals():
    raise RuntimeError("Ejecuta C04 primero: faltan los snapshots validados.")
if "banrep" not in globals():
    raise RuntimeError("Ejecuta C04 primero: falta el CSV BanRep validado.")

query_time = datetime.now(timezone.utc).isoformat()
results = []

def add_not_verified(source, exc):
    results.append({
        "consulta_utc": query_time,
        "fuente": source,
        "estado": "no verificada",
        "detalle": f"{type(exc).__name__}: {exc}",
    })

def compare(source, reference, live, ref_value, live_value):
    ref = reference[["date", ref_value]].copy()
    current = live[["date", live_value]].copy()
    ref["date"] = pd.to_datetime(ref["date"], errors="coerce").dt.normalize()
    current["date"] = pd.to_datetime(current["date"], errors="coerce").dt.normalize()
    ref[ref_value] = pd.to_numeric(ref[ref_value], errors="coerce")
    current[live_value] = pd.to_numeric(current[live_value], errors="coerce")
    ref = ref.dropna().drop_duplicates("date")
    current = current.dropna().drop_duplicates("date")
    merged = ref.merge(current, on="date", how="outer", indicator=True)
    common = merged.loc[merged["_merge"].eq("both")].copy()
    delta = (
        pd.to_numeric(common[ref_value], errors="coerce")
        - pd.to_numeric(common[live_value], errors="coerce")
    ).abs()
    different = int((delta > 1e-8).sum())
    only_ref = int(merged["_merge"].eq("left_only").sum())
    only_live = int(merged["_merge"].eq("right_only").sum())
    status = (
        "verificada; sin discrepancias"
        if different == 0 and only_ref == 0 and only_live == 0
        else "verificada; discrepancias detectadas"
    )
    return {
        "consulta_utc": query_time,
        "fuente": source,
        "estado": status,
        "filas_snapshot": len(ref),
        "filas_descarga": len(current),
        "fechas_comunes": len(common),
        "solo_snapshot": only_ref,
        "solo_descarga": only_live,
        "valores_diferentes": different,
        "diferencia_absoluta_maxima": float(delta.max()) if len(delta) else None,
        "detalle": "La descarga verifica; el snapshot sigue siendo primario.",
    }

def run_check(source, function):
    try:
        results.append(function())
    except Exception as exc:
        add_not_verified(source, exc)

def verify_trm():
    response = requests.get(
        "https://www.datos.gov.co/resource/32sa-8pi3.json",
        params={
            "$limit": 50000,
            "$order": "vigenciadesde",
            "$where": "vigenciadesde >= '1992-01-01T00:00:00'",
        },
        timeout=60,
    )
    response.raise_for_status()
    raw = pd.DataFrame(response.json())
    if not {"vigenciadesde", "valor"}.issubset(raw.columns):
        raise ValueError("La respuesta TRM no incluye vigenciadesde y valor.")
    live = pd.DataFrame({
        "date": pd.to_datetime(
            raw["vigenciadesde"], errors="coerce", utc=True
        ).dt.tz_localize(None).dt.normalize(),
        "trm_live": pd.to_numeric(raw["valor"], errors="coerce"),
    }).dropna()
    reference = trm_raw.rename(columns={"fecha": "date", "trm": "trm_snapshot"})
    return compare("datos.gov.co — TRM", reference, live, "trm_snapshot", "trm_live")

def verify_market_ticker(value_col, ticker):
    start = pd.to_datetime(market["date"]).min().date().isoformat()
    end = (pd.to_datetime(market["date"]).max() + pd.Timedelta(days=1)).date().isoformat()
    data = yf.download(
        ticker, start=start, end=end, interval="1d",
        auto_adjust=False, progress=False, threads=False,
    )
    if data.empty:
        raise RuntimeError(f"Yahoo Finance devolvió vacío para {ticker}.")
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    if "Close" not in data.columns:
        raise ValueError(f"No hay columna Close para {ticker}.")
    dates = pd.to_datetime(data.index, utc=True).tz_localize(None).normalize()
    live = pd.DataFrame({
        "date": dates,
        "close_live": pd.to_numeric(data["Close"], errors="coerce").to_numpy(),
    }).dropna()
    reference = market[["date", value_col]].rename(columns={value_col: "close_snapshot"})
    return compare(
        f"Yahoo Finance — {ticker}", reference, live,
        "close_snapshot", "close_live",
    )

def verify_embi():
    end = pd.to_datetime(embi["date"]).max()
    url = (
        "https://estadisticas.bcrp.gob.pe/estadisticas/series/api/"
        f"PD04715XD/json/1998-1-2/{end.year}-{end.month}-{end.day}/ing"
    )
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    periods = response.json().get("periods", [])
    if not periods:
        raise ValueError("BCRPData no devolvió periodos.")
    months = {
        "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
        "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
        "Ene": 1, "Abr": 4, "Ago": 8, "Set": 9, "Dic": 12,
    }
    parsed = []
    for item in periods:
        match = re.fullmatch(
            r"(\d{1,2})\.([A-Za-z]{3})\.(\d{2})",
            str(item.get("name", "")).strip(),
        )
        if not match or match.group(2) not in months:
            continue
        day, month, short_year = int(match.group(1)), months[match.group(2)], int(match.group(3))
        year = 2000 + short_year if short_year < 70 else 1900 + short_year
        parsed.append({
            "date": pd.Timestamp(year=year, month=month, day=day),
            "embi_live": pd.to_numeric(item.get("values", [None])[0], errors="coerce"),
        })
    live = pd.DataFrame(parsed).dropna()
    reference = embi.rename(columns={"embi_colombia": "embi_snapshot"})
    return compare("BCRPData — EMBI Colombia", reference, live, "embi_snapshot", "embi_live")

def verify_banrep():
    url = (
        "https://suameca.banrep.gov.co/estadisticas-economicas-back/rest/"
        "estadisticaEconomicaRestService/consultaMenuXId?idMenu=59"
    )
    response = requests.get(
        url, timeout=60,
        headers={"Accept": "application/json", "User-Agent": "ML2-project/1.0"},
    )
    response.raise_for_status()
    series = next(
        (item for item in response.json().get("SERIES", [])
         if str(item.get("id")) == "59"),
        None,
    )
    if series is None or not series.get("data"):
        raise ValueError("La API no devolvió la serie 59.")
    if str(series.get("unidadCorta", "")).strip() != "%":
        raise ValueError("La unidad de la API no es porcentaje.")
    raw = pd.DataFrame(series["data"], columns=["timestamp_ms", "tasa_api"])
    raw["date"] = pd.to_datetime(
        pd.to_numeric(raw["timestamp_ms"], errors="coerce"),
        unit="ms", errors="coerce", utc=True,
    ).dt.tz_localize(None).dt.normalize()
    raw["tasa_api"] = pd.to_numeric(raw["tasa_api"], errors="coerce")
    api = raw.dropna(subset=["date", "tasa_api"]).sort_values("date")
    changes = banrep[["fecha_efectiva", "tasa_intervencion"]].rename(
        columns={"fecha_efectiva": "effective_date", "tasa_intervencion": "tasa_csv"}
    ).sort_values("effective_date")
    aligned = pd.merge_asof(
        api, changes, left_on="date", right_on="effective_date", direction="backward"
    )
    common = aligned.dropna(subset=["tasa_csv"])
    delta = (common["tasa_api"] - common["tasa_csv"]).abs()
    different = int((delta > 1e-8).sum())
    status = (
        "verificada; sin discrepancias"
        if different == 0 and len(common) == len(api)
        else "verificada; discrepancias detectadas"
    )
    return {
        "consulta_utc": query_time,
        "fuente": "BanRep API frente al CSV oficial de cambios",
        "estado": status,
        "filas_snapshot": len(banrep),
        "filas_descarga": len(api),
        "fechas_comunes": len(common),
        "solo_snapshot": None,
        "solo_descarga": int(len(api) - len(common)),
        "valores_diferentes": different,
        "diferencia_absoluta_maxima": float(delta.max()) if len(delta) else None,
        "detalle": "El CSV BanRep permanece como insumo primario.",
    }

run_check("datos.gov.co — TRM", verify_trm)
for value_col, ticker in {
    "brent_close": "BZ=F",
    "wti_close": "CL=F",
    "dxy_close": "DX-Y.NYB",
    "mxn_close": "MXN=X",
    "clp_close": "CLP=X",
    "pen_close": "PEN=X",
}.items():
    run_check(
        f"Yahoo Finance — {ticker}",
        lambda value_col=value_col, ticker=ticker:
            verify_market_ticker(value_col, ticker),
    )
run_check("BCRPData — EMBI Colombia", verify_embi)
run_check("BanRep API", verify_banrep)

verification_df = pd.DataFrame(results)
verification_path = Path(RUN_DIR) / "verificacion_fuentes.csv"
verification_df.to_csv(verification_path, index=False)
print("Estado de verificación por fuente:")
print(verification_df[["fuente", "estado"]].to_string(index=False))
print(f"Detalle guardado en: {verification_path}")
print("Las consultas opcionales no reemplazan ningún archivo primario.")


In [ ]:
# C07 — Reconstrucción desde el snapshot TRM primario
from pathlib import Path
import numpy as np
import pandas as pd

if "PROJECT_DIR" not in globals() or not Path(PROJECT_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: PROJECT_DIR no está definido.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
if "trm_raw" not in globals() or not isinstance(trm_raw, pd.DataFrame):
    raise RuntimeError("Ejecuta C04 primero: falta trm_raw.")
if "trm_reference" not in globals() or not isinstance(trm_reference, pd.DataFrame):
    raise RuntimeError("Ejecuta C04 primero: falta trm_spikes_dataset.csv.")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
required_raw = {"fecha", "trm"}
if not required_raw.issubset(trm_raw.columns):
    raise ValueError(f"Snapshot TRM incompleto: {required_raw - set(trm_raw.columns)}.")

filas_snapshot_crudo = len(trm_raw)
print(f"Filas disponibles en snapshot crudo: {filas_snapshot_crudo:,}")

trm_valida = trm_raw[["fecha", "trm"]].copy()
trm_valida["fecha"] = pd.to_datetime(
    trm_valida["fecha"], errors="coerce"
).dt.normalize()
trm_valida["trm"] = pd.to_numeric(trm_valida["trm"], errors="coerce")
trm_valida = (
    trm_valida.dropna(subset=["fecha", "trm"])
    .loc[lambda d: d["trm"] > 0]
    .sort_values("fecha")
    .drop_duplicates("fecha")
    .reset_index(drop=True)
)
if trm_valida.empty:
    raise ValueError("La reconstrucción dejó la TRM vacía.")
if trm_valida["fecha"].duplicated().any() or not trm_valida["fecha"].is_monotonic_increasing:
    raise AssertionError("La TRM debe tener fechas únicas y ordenadas.")
if len(trm_valida) != 8364:
    raise AssertionError(f"TRM válida esperada 8.364; obtenida {len(trm_valida):,}.")
print(f"TRM válida: {len(trm_valida):,}")

trm_retornos = trm_valida.copy()
trm_retornos["log_return"] = np.log(trm_retornos["trm"] / trm_retornos["trm"].shift(1))
trm_retornos = trm_retornos.dropna(subset=["log_return"]).reset_index(drop=True)
if len(trm_retornos) != 8363:
    raise AssertionError(f"Retornos esperados 8.363; obtenidos {len(trm_retornos):,}.")
print(f"Retornos: {len(trm_retornos):,}")

def comparar_trm_referencia(referencia, nombre, columna_fecha):
    requeridas = {columna_fecha, "trm", "log_return"}
    if not requeridas.issubset(referencia.columns):
        raise ValueError(f"{nombre}: faltan columnas {requeridas - set(referencia.columns)}.")
    ref = referencia[[columna_fecha, "trm", "log_return"]].copy()
    ref = ref.rename(columns={columna_fecha: "fecha"})
    ref["fecha"] = pd.to_datetime(ref["fecha"], errors="coerce").dt.normalize()
    ref["trm"] = pd.to_numeric(ref["trm"], errors="coerce")
    ref["log_return"] = pd.to_numeric(ref["log_return"], errors="coerce")
    if ref[["fecha", "trm", "log_return"]].isna().any().any():
        raise ValueError(f"{nombre}: fechas o cifras inválidas.")
    cmp = trm_retornos[["fecha", "trm", "log_return"]].merge(
        ref, on="fecha", how="inner",
        suffixes=("_reconstruida", "_referencia"), validate="one_to_one",
    )
    if cmp.empty:
        raise AssertionError(f"No hay fechas comunes con {nombre}.")
    trm_ok = np.allclose(cmp.trm_reconstruida, cmp.trm_referencia, rtol=1e-9, atol=1e-10)
    ret_ok = np.allclose(
        cmp.log_return_reconstruida, cmp.log_return_referencia,
        rtol=1e-9, atol=1e-10,
    )
    print(f"{nombre}: fechas comunes={len(cmp):,}; TRM coincide={trm_ok}; retornos coinciden={ret_ok}")
    if not trm_ok or not ret_ok:
        raise AssertionError(f"La TRM reconstruida difiere de {nombre}.")

processed_path = Path(SNAPSHOT_DIR) / "trm_aprobada_fase1_20261112T055009052854Z.csv"
if not processed_path.is_file():
    raise FileNotFoundError(f"Falta la referencia procesada: {processed_path}")
trm_aprobada_ref = pd.read_csv(processed_path)
comparar_trm_referencia(trm_aprobada_ref, "snapshot procesado", "date")
comparar_trm_referencia(trm_reference, "trm_spikes_dataset.csv", "fecha")
print("C07 compara fecha/TRM/retorno; etiquetas y rezagos se comparan en C14.")


In [ ]:
# C08 — Ljung–Box sobre retornos
from pathlib import Path
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox

if "trm_retornos" not in globals() or not isinstance(trm_retornos, pd.DataFrame):
    raise RuntimeError("Ejecuta C07 primero: falta trm_retornos.")
if "log_return" not in trm_retornos.columns or len(trm_retornos) < 40:
    raise ValueError("Se necesitan al menos 40 retornos.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")

def diagnostico_ljung_box(serie_retornos, lags=(1, 5, 11, 21, 36)):
    resultado = acorr_ljungbox(
        serie_retornos.astype(float), lags=list(lags), return_df=True
    )
    print("Ljung–Box sobre retornos (H0: no hay autocorrelación serial):")
    print(resultado.round(4))
    rechazadas = resultado[resultado["lb_pvalue"] < 0.01]
    if len(rechazadas):
        print(f"Se rechaza H0 al 1% en {len(rechazadas)} de {len(resultado)} rezagos.")
    else:
        print("No se rechaza H0 en los rezagos probados.")
    return resultado

lb_retornos = diagnostico_ljung_box(trm_retornos["log_return"])
lb_retornos.to_csv(Path(RUN_DIR) / "ljung_box_retornos.csv", index=True)


In [ ]:
# C09 — Búsqueda AR-GARCH/EGARCH (108 especificaciones) con checkpoint
from pathlib import Path
import hashlib
import json
import os
import pickle
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from arch import arch_model

if "trm_retornos" not in globals() or not isinstance(trm_retornos, pd.DataFrame):
    raise RuntimeError("Ejecuta C07 primero: falta trm_retornos.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
if not {"fecha", "log_return"}.issubset(trm_retornos.columns):
    raise ValueError("trm_retornos debe contener fecha y log_return.")
if trm_retornos["fecha"].duplicated().any():
    raise ValueError("No se puede ajustar GARCH con fechas duplicadas.")

# Reasigna trm_retornos con índice de fechas; C10 depende de ese índice.
# La bandera y el helper se definen siempre, también al reanudar checkpoints.
MODELO_GARCH_VALIDADO = False
def validar_conteo_modelo(actual, esperado, etapa):
    if actual == esperado:
        print(f"{etapa}: {actual:,} filas; coincide con la referencia.")
        return
    mensaje = f"{etapa}: esperado {esperado:,}; obtenido {actual:,}."
    if MODELO_GARCH_VALIDADO:
        raise AssertionError(mensaje)
    warnings.warn(mensaje + " Modelo distinto del validado; se continúa.", RuntimeWarning)

# Conteos de referencia Fase 2: verificaciones informativas, nunca bloqueantes.
CONTEOS_REFERENCIA_FASE2 = {
    "panel_filas": 6835, "panel_spikes": 338,
    "train_filas": 6151, "test_filas": 684, "test_spikes": 33,
}

def verificar_conteo_fase2(actual, clave):
    if clave not in CONTEOS_REFERENCIA_FASE2:
        raise KeyError(f"Conteo sin referencia documentada: {clave}.")
    esperado = CONTEOS_REFERENCIA_FASE2[clave]
    if int(actual) == esperado:
        print(f"Verificacion Fase 2 {clave}: {int(actual):,}; coincide.")
        return True
    mensaje = f"{clave}: esperado {esperado:,}; obtenido {int(actual):,}. Se advierte y se continua."
    warnings.warn(mensaje, RuntimeWarning)
    print("ADVERTENCIA:", mensaje)
    return False

trm_retornos = trm_retornos.copy()
trm_retornos["fecha"] = pd.to_datetime(trm_retornos["fecha"], errors="raise").dt.normalize()
trm_retornos = trm_retornos.set_index("fecha", drop=False).rename_axis(None)
split_raw = int(len(trm_retornos) * 0.90)
if split_raw <= 30 or split_raw >= len(trm_retornos):
    raise ValueError(f"Corte temporal inválido: split_raw={split_raw}.")
train_retornos = trm_retornos["log_return"].iloc[:split_raw].astype(float)
if not train_retornos.index.is_monotonic_increasing:
    raise AssertionError("El entrenamiento GARCH debe conservar orden temporal.")

MODO_RAPIDO = bool(globals().get("MODO_RAPIDO", False))
if MODO_RAPIDO:
    # Subconjunto diagnóstico que incluye las especificaciones AR(3) y AR(11) EGARCH-t.
    AR_LAGS = (3, 11)
    VOL_MODELS = ("EGARCH",)
    PQ_GRID = ((1, 1), (2, 1))
    DISTRIBUTIONS = ("t",)
    N_ESPECIFICACIONES_GARCH = 4
else:
    AR_LAGS = (1, 3, 5, 7, 11, 15)
    VOL_MODELS = ("GARCH", "EGARCH")
    PQ_GRID = ((1, 1), (1, 2), (2, 1))
    DISTRIBUTIONS = ("normal", "t", "ged")
    N_ESPECIFICACIONES_GARCH = 108
specs = [
    {"ar_lags": ar_lags, "vol_model": vol_model, "p": p, "q": q, "dist": dist}
    for ar_lags in AR_LAGS for vol_model in VOL_MODELS
    for p, q in PQ_GRID for dist in DISTRIBUTIONS
]
if len(specs) != N_ESPECIFICACIONES_GARCH:
    raise AssertionError(
        f"Grilla esperada {N_ESPECIFICACIONES_GARCH}; obtenidas {len(specs)}."
    )
for spec in specs:
    spec["spec_id"] = f"AR{spec['ar_lags']}_{spec['vol_model']}_{spec['p']}_{spec['q']}_{spec['dist']}"

data_digest = hashlib.sha256()
data_digest.update(pd.DatetimeIndex(train_retornos.index).asi8.tobytes())
data_digest.update(np.asarray(train_retornos.to_numpy(), dtype="<f8").tobytes())
input_sha256 = data_digest.hexdigest()
grid_json = json.dumps(
    [{k: v for k, v in spec.items() if k != "spec_id"} for spec in specs],
    sort_keys=True,
)
grid_sha256 = hashlib.sha256(grid_json.encode("utf-8")).hexdigest()
checkpoint_path = Path(RUN_DIR) / "garch_fase1_checkpoint.csv"
fit_dir = Path(RUN_DIR) / "garch_fase1_fits"
fit_dir.mkdir(parents=True, exist_ok=True)
completed = {}

if checkpoint_path.exists():
    old = pd.read_csv(checkpoint_path)
    needed = {"spec_id", "status", "input_sha256", "grid_sha256"}
    if not needed.issubset(old.columns):
        raise ValueError(f"Checkpoint incompleto: {checkpoint_path}")
    if (
        old["input_sha256"].nunique() != 1
        or old["input_sha256"].iloc[0] != input_sha256
        or old["grid_sha256"].nunique() != 1
        or old["grid_sha256"].iloc[0] != grid_sha256
    ):
        raise RuntimeError("Checkpoint de datos o grilla distinta; no se reutiliza.")
    completed = {row["spec_id"]: row.to_dict() for _, row in old.iterrows()}

def guardar_checkpoint():
    temp_path = checkpoint_path.with_suffix(".tmp.csv")
    pd.DataFrame(list(completed.values())).to_csv(temp_path, index=False)
    os.replace(temp_path, checkpoint_path)

for spec in specs:
    spec_id = spec["spec_id"]
    fit_path = fit_dir / f"{spec_id}.pkl"
    previous = completed.get(spec_id)
    if previous is not None:
        previous_fit = Path(str(previous.get("fit_path", "")))
        if previous.get("status") != "convergente" or previous_fit.is_file():
            continue
    try:
        am = arch_model(
            train_retornos * 100, mean="AR", lags=spec["ar_lags"],
            vol=spec["vol_model"], p=spec["p"], q=spec["q"], dist=spec["dist"],
        )
        res = am.fit(disp="off", show_warning=False)
        if res.convergence_flag == 0:
            temp_fit_path = fit_path.with_suffix(".tmp")
            with open(temp_fit_path, "wb") as stream:
                pickle.dump(res, stream, protocol=pickle.HIGHEST_PROTOCOL)
            os.replace(temp_fit_path, fit_path)
            status, bic, error, saved_fit = "convergente", float(res.bic), "", str(fit_path)
        else:
            status, bic = "no convergió", np.nan
            error, saved_fit = f"convergence_flag={res.convergence_flag}", ""
    except Exception as exc:
        status, bic = "error", np.nan
        error, saved_fit = f"{type(exc).__name__}: {exc}"[:1000], ""

    completed[spec_id] = {
        **{k: spec[k] for k in ("spec_id", "ar_lags", "vol_model", "p", "q", "dist")},
        "status": status, "bic": bic, "error": error, "fit_path": saved_fit,
        "input_sha256": input_sha256, "grid_sha256": grid_sha256,
    }
    guardar_checkpoint()
    print(f"{spec_id}: {status}; checkpoint actualizado.")

if len(completed) != N_ESPECIFICACIONES_GARCH:
    raise AssertionError(f"Checkpoint contiene {len(completed)} de {N_ESPECIFICACIONES_GARCH} especificaciones.")
checkpoint_df = pd.DataFrame(list(completed.values()))
if checkpoint_df["spec_id"].nunique() != N_ESPECIFICACIONES_GARCH:
    raise AssertionError("Hay especificaciones duplicadas en el checkpoint.")
tabla_resultados = checkpoint_df.loc[
    checkpoint_df["status"].eq("convergente")
].copy()
if tabla_resultados.empty:
    raise RuntimeError("Ninguna especificación GARCH convergió.")
tabla_resultados = tabla_resultados.sort_values("bic", kind="stable").reset_index(drop=True)
mejor = tabla_resultados.iloc[0]
bic_mejor = float(mejor["bic"])
n_convergentes = int(checkpoint_df["status"].eq("convergente").sum())
print(f"Especificaciones convergentes: {n_convergentes} de {N_ESPECIFICACIONES_GARCH}.")
if not MODO_RAPIDO and n_convergentes != 94:
    warnings.warn(f"Se esperaban 94 convergentes de 108; hubo {n_convergentes}.", RuntimeWarning)

especificacion_mejor = {
    "ar_lags": int(mejor["ar_lags"]), "vol_model": str(mejor["vol_model"]),
    "p": int(mejor["p"]), "q": int(mejor["q"]), "dist": str(mejor["dist"]),
}
ESPECIFICACION_GARCH_VALIDADA = (
    especificacion_mejor["ar_lags"] == 3
    and especificacion_mejor["vol_model"].lower() == "egarch"
    and especificacion_mejor["p"] == 2
    and especificacion_mejor["q"] == 1
    and especificacion_mejor["dist"].lower() == "t"
)
BIC_GARCH_VALIDADO = abs(bic_mejor - 9792.14) <= 0.05
MODELO_GARCH_VALIDADO = (not MODO_RAPIDO) and ESPECIFICACION_GARCH_VALIDADA and BIC_GARCH_VALIDADO
print(f"¿Coincide la especificación AR(3)-EGARCH(2,1)-t?: {ESPECIFICACION_GARCH_VALIDADA}")
print(f"¿BIC coincide con 9792,14 (tolerancia 0,05)?: {BIC_GARCH_VALIDADO} (obtenido {bic_mejor:.2f})")
print(f"¿Coincide el modelo completo con la referencia en modo completo?: {MODELO_GARCH_VALIDADO}")
if MODO_RAPIDO:
    print("Modo rápido: el subconjunto no certifica el mínimo global; no se marca como modelo validado.")

fit_path = Path(str(mejor["fit_path"]))
try:
    with open(fit_path, "rb") as stream:
        ajuste = pickle.load(stream)
    if not np.isclose(float(ajuste.bic), bic_mejor, rtol=0, atol=1e-10):
        raise ValueError("BIC del objeto cargado distinto del checkpoint.")
except Exception as exc:
    aviso = (
        f"{datetime.now(timezone.utc).isoformat()} | "
        f"No se pudo cargar {fit_path.name}: {type(exc).__name__}: {exc}. "
        "Se reajusta solo la especificación ganadora."
    )
    warnings.warn(aviso, RuntimeWarning)
    with open(Path(RUN_DIR) / "avisos_checkpoint_garch.txt", "a", encoding="utf-8") as stream:
        stream.write(aviso + "\n")
    am = arch_model(
        train_retornos * 100, mean="AR", lags=especificacion_mejor["ar_lags"],
        vol=especificacion_mejor["vol_model"], p=especificacion_mejor["p"],
        q=especificacion_mejor["q"], dist=especificacion_mejor["dist"],
    )
    ajuste = am.fit(disp="off", show_warning=False)
    if ajuste.convergence_flag != 0:
        raise RuntimeError(f"Reajuste ganador no convergió: {ajuste.convergence_flag}.")
    temp_fit_path = fit_path.with_suffix(".tmp")
    with open(temp_fit_path, "wb") as stream:
        pickle.dump(ajuste, stream, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp_fit_path, fit_path)

tabla_resultados[["ar_lags", "vol_model", "p", "q", "dist", "bic"]].to_csv(
    Path(RUN_DIR) / "garch_fase1_resultados.csv", index=False
)


In [ ]:
# C10 — Etiquetado GARCH causal y registro de filas excluidas
import numpy as np
import pandas as pd

if "trm_retornos" not in globals() or not isinstance(trm_retornos, pd.DataFrame):
    raise RuntimeError("Ejecuta C07 y C09 primero: falta trm_retornos.")
if "ajuste" not in globals() or "split_raw" not in globals():
    raise RuntimeError("Ejecuta C09 primero: falta ajuste o split_raw.")
if "validar_conteo_modelo" not in globals() or "MODELO_GARCH_VALIDADO" not in globals():
    raise RuntimeError("C09 debe definir validar_conteo_modelo y MODELO_GARCH_VALIDADO.")
if not isinstance(trm_retornos.index, pd.DatetimeIndex):
    raise TypeError("trm_retornos debe tener índice de fechas.")
if not trm_retornos.index.is_unique:
    raise ValueError("trm_retornos tiene fechas duplicadas.")

etiqueta_intermedia = None
def definir_etiquetas(df, ajuste, split_raw):
    global etiqueta_intermedia
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice debe contener fechas para etiquetar.")
    if not df.index.is_unique or not df.index.equals(pd.DatetimeIndex(df["fecha"])):
        raise AssertionError("El índice y la columna fecha no están alineados.")

    df["label_source_date"] = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    par = ajuste.params
    df["cond_sd"] = np.nan
    df["resid"] = np.nan
    fit_dates = pd.DatetimeIndex(ajuste.conditional_volatility.index)
    resid_dates = pd.DatetimeIndex(ajuste.resid.index)
    if not fit_dates.equals(resid_dates) or not fit_dates.isin(df.index).all():
        raise AssertionError("Las fechas de residuos/volatilidad GARCH no coinciden con TRM.")
    assert pd.DatetimeIndex(pd.to_datetime(df.loc[fit_dates, "fecha"])).equals(fit_dates)
    df.loc[fit_dates, "cond_sd"] = ajuste.conditional_volatility.to_numpy() / 100
    df.loc[fit_dates, "resid"] = ajuste.resid.to_numpy() / 100
    df.loc[fit_dates, "label_source_date"] = fit_dates

    ar_coefs = {}
    for nombre, valor in par.items():
        nombre = str(nombre)
        if nombre.startswith("log_return[") and nombre.endswith("]"):
            lag = int(nombre[len("log_return["):-1])
            ar_coefs[lag] = float(valor)

    def coeficientes_volatilidad(prefijo):
        encontrados = []
        for nombre, valor in par.items():
            nombre = str(nombre)
            if nombre.startswith(prefijo + "[") and nombre.endswith("]"):
                orden = int(nombre[len(prefijo) + 1:-1])
                encontrados.append((orden, float(valor)))
        return [valor for _, valor in sorted(encontrados)]

    omega = float(par.get("omega", 0.0))
    alpha = coeficientes_volatilidad("alpha")
    beta = coeficientes_volatilidad("beta")
    gamma = coeficientes_volatilidad("gamma")
    nombre_vol = ajuste.model.volatility.name.lower()
    historial = pd.concat(
        [
            ajuste.resid.rename("eps"),
            ajuste.conditional_volatility.rename("sigma"),
        ],
        axis=1,
    ).dropna()
    eps_hist = historial["eps"].astype(float).tolist()[::-1]
    sigma_hist = historial["sigma"].astype(float).tolist()[::-1]

    for pos in range(split_raw, len(df)):
        fecha_t = df.index[pos]
        if nombre_vol == "egarch":
            log_h = omega
            for j, coef in enumerate(alpha):
                if j < len(eps_hist) and j < len(sigma_hist):
                    z = eps_hist[j] / max(sigma_hist[j], 1e-12)
                    log_h += coef * (abs(z) - np.sqrt(2 / np.pi))
            for j, coef in enumerate(gamma):
                if j < len(eps_hist) and j < len(sigma_hist):
                    z = eps_hist[j] / max(sigma_hist[j], 1e-12)
                    log_h += coef * z
            for j, coef in enumerate(beta):
                if j < len(sigma_hist):
                    log_h += coef * np.log(max(sigma_hist[j] ** 2, 1e-12))
            sigma_t = np.sqrt(np.exp(np.clip(log_h, -30, 30)))
        else:
            h = omega
            for j, coef in enumerate(alpha):
                if j < len(eps_hist):
                    h += coef * eps_hist[j] ** 2
            for j, coef in enumerate(gamma):
                if j < len(eps_hist):
                    h += coef * eps_hist[j] ** 2 * (eps_hist[j] < 0)
            for j, coef in enumerate(beta):
                if j < len(sigma_hist):
                    h += coef * sigma_hist[j] ** 2
            sigma_t = np.sqrt(max(h, 1e-12))

        mu = float(par.get("Const", 0.0))
        for lag, coef in sorted(ar_coefs.items()):
            mu += coef * float(df["log_return"].iloc[pos - lag]) * 100
        eps_t = float(df["log_return"].iloc[pos] * 100 - mu)
        df.loc[fecha_t, "cond_sd"] = sigma_t / 100
        df.loc[fecha_t, "resid"] = eps_t / 100
        df.loc[fecha_t, "label_source_date"] = fecha_t
        eps_hist.insert(0, eps_t)
        sigma_hist.insert(0, sigma_t)

    df["spike"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    validos = df["cond_sd"].notna() & df["resid"].notna()
    label_dates = pd.DatetimeIndex(df.index[validos.to_numpy()])
    assert pd.DatetimeIndex(pd.to_datetime(df.loc[label_dates, "fecha"])).equals(label_dates)
    spike_values = (
        df.loc[label_dates, "resid"].abs()
        > 2 * df.loc[label_dates, "cond_sd"]
    ).astype(int)
    df.loc[label_dates, "spike"] = spike_values.to_numpy()
    df.loc[label_dates, "label_source_date"] = label_dates

    etiqueta_intermedia = df.copy()
    salida = df.dropna(
        subset=["cond_sd", "resid", "spike", "label_source_date"]
    )
    salida = salida.astype({"spike": int})
    assert pd.DatetimeIndex(pd.to_datetime(salida["fecha"])).equals(
        pd.DatetimeIndex(pd.to_datetime(salida["label_source_date"]))
    )
    return salida.reset_index(drop=True)

trm_etiquetada = definir_etiquetas(trm_retornos, ajuste, split_raw)
validar_conteo_modelo(len(trm_etiquetada), 8360, "Serie etiquetada GARCH")
print(f"Spikes: {int(trm_etiquetada['spike'].sum()):,}")
print(f"Prevalencia: {trm_etiquetada['spike'].mean():.4%}")

filas_excluidas = etiqueta_intermedia.loc[
    etiqueta_intermedia[
        ["cond_sd", "resid", "spike", "label_source_date"]
    ].isna().any(axis=1)
].copy()
if len(filas_excluidas) != len(trm_retornos) - len(trm_etiquetada):
    raise AssertionError("Las filas excluidas no explican el cambio de conteo.")
nulos_por_fila = (
    etiqueta_intermedia[
        ["cond_sd", "resid", "spike", "label_source_date"]
    ]
    .isna()
    .apply(lambda row: ", ".join(row.index[row].tolist()), axis=1)
)
filas_excluidas["columnas_nulas"] = nulos_por_fila.loc[filas_excluidas.index]
print("Filas sin etiqueta; fecha y columnas nulas:")
print(filas_excluidas[["fecha", "columnas_nulas"]].to_string(index=False))


In [ ]:
# C11 — Verificación de etiquetas por fecha
import pandas as pd

if "trm_etiquetada" not in globals():
    raise RuntimeError("Ejecuta C10 primero: falta trm_etiquetada.")
if "validar_conteo_modelo" not in globals():
    raise RuntimeError("Ejecuta C09 primero: falta validar_conteo_modelo.")
required = {"fecha", "label_source_date", "spike"}
if not required.issubset(trm_etiquetada.columns):
    raise ValueError(f"Faltan columnas: {required - set(trm_etiquetada.columns)}.")

fechas_etiqueta = pd.DatetimeIndex(pd.to_datetime(trm_etiquetada["fecha"], errors="raise"))
fechas_fuente = pd.DatetimeIndex(pd.to_datetime(trm_etiquetada["label_source_date"], errors="raise"))
assert fechas_etiqueta.is_unique, "Hay fechas duplicadas en la serie etiquetada."
assert fechas_etiqueta.equals(fechas_fuente), "Etiqueta y fecha fuente no coinciden."
assert trm_etiquetada["spike"].notna().all()
assert trm_etiquetada["spike"].isin([0, 1]).all()
validar_conteo_modelo(len(trm_etiquetada), 8360, "Serie etiquetada verificada")
print(f"Correspondencia fecha-etiqueta aprobada: {len(trm_etiquetada):,} filas.")
trm_etiquetada = trm_etiquetada.drop(columns=["label_source_date"])


In [ ]:
# C12 — Diagnósticos del entrenamiento y ajuste GARCH
from pathlib import Path
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

if "train_retornos" not in globals() or "ajuste" not in globals():
    raise RuntimeError("Ejecuta C09 primero: faltan train_retornos o ajuste.")
if len(train_retornos.dropna()) < 30:
    raise ValueError("Se requieren al menos 30 retornos de entrenamiento.")
if not hasattr(ajuste, "std_resid"):
    raise TypeError("El ajuste no expone residuos estandarizados.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")

retornos_diag = train_retornos.dropna().astype(float)
adf_stat, adf_p, adf_lags, adf_n, adf_crit, _ = adfuller(retornos_diag, autolag="AIC")
kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(
    retornos_diag, regression="c", nlags="auto"
)
residuos_est = pd.Series(ajuste.std_resid).dropna().astype(float)
lags_lb = [lag for lag in (10, 20) if lag < len(residuos_est)]
if not lags_lb:
    raise ValueError("No hay residuos suficientes para Ljung–Box en lags 10 y 20.")
lb_res = acorr_ljungbox(residuos_est, lags=lags_lb, return_df=True)
lb_sq = acorr_ljungbox(residuos_est.pow(2), lags=lags_lb, return_df=True)
arch_lags = min(10, len(residuos_est) - 1)
arch_lm, arch_p, arch_f, arch_f_p = het_arch(residuos_est, nlags=arch_lags)

diagnosticos = [
    {"prueba": "ADF retornos train", "estadistico": adf_stat, "p_valor": adf_p, "rezagos": adf_lags},
    {"prueba": "KPSS retornos train", "estadistico": kpss_stat, "p_valor": kpss_p, "rezagos": kpss_lags},
]
for lag, row in lb_res.iterrows():
    diagnosticos.append({
        "prueba": f"Ljung–Box residuos estandarizados (lag {lag})",
        "estadistico": row["lb_stat"], "p_valor": row["lb_pvalue"], "rezagos": lag,
    })
for lag, row in lb_sq.iterrows():
    diagnosticos.append({
        "prueba": f"Ljung–Box residuos estandarizados^2 (lag {lag})",
        "estadistico": row["lb_stat"], "p_valor": row["lb_pvalue"], "rezagos": lag,
    })
diagnosticos.extend([
    {"prueba": "ARCH-LM residuos estandarizados", "estadistico": arch_lm, "p_valor": arch_p, "rezagos": arch_lags},
    {"prueba": "ARCH-LM F", "estadistico": arch_f, "p_valor": arch_f_p, "rezagos": arch_lags},
])
tabla_diagnosticos = pd.DataFrame(diagnosticos)
print(tabla_diagnosticos.to_string(
    index=False,
    formatters={"estadistico": "{:.4f}".format, "p_valor": "{:.4g}".format},
))
print("KPSS p=0,10 puede ser el límite superior tabulado, no una probabilidad exacta.")
if (lb_res["lb_pvalue"] < 0.05).any():
    print("Ljung–Box rechaza en residuos estandarizados: queda autocorrelación en la media.")
else:
    print("Ljung–Box no rechaza autocorrelación en residuos estandarizados.")
if (lb_sq["lb_pvalue"] < 0.05).any() or arch_p < 0.05:
    print("Los diagnósticos sugieren dependencia de varianza residual.")
else:
    print("Los cuadrados y ARCH-LM no detectan heterocedasticidad residual.")
tabla_diagnosticos.to_csv(Path(RUN_DIR) / "diagnosticos_garch_fase1.csv", index=False)


In [ ]:
# C13 — Rezagos, objetivo del día siguiente y partición temporal
import pandas as pd

if "trm_etiquetada" not in globals():
    raise RuntimeError("Ejecuta C11 primero: falta la serie etiquetada.")
if "validar_conteo_modelo" not in globals():
    raise RuntimeError("Ejecuta C09 primero: falta validar_conteo_modelo.")
required = {"fecha", "trm", "log_return", "spike"}
if not required.issubset(trm_etiquetada.columns):
    raise ValueError(f"Faltan columnas: {required - set(trm_etiquetada.columns)}.")
if trm_etiquetada["fecha"].duplicated().any():
    raise ValueError("No se pueden crear rezagos con fechas duplicadas.")

def split_temporal(df, feature_cols, target_col="target_next_day", pct_train=0.90):
    if not 0 < pct_train < 1:
        raise ValueError("pct_train debe estar entre 0 y 1.")
    if not df["fecha"].is_monotonic_increasing:
        raise ValueError("La partición requiere orden cronológico.")
    split_idx = int(len(df) * pct_train)
    X, y = df[feature_cols].copy(), df[target_col].astype(int).copy()
    return (
        X.iloc[:split_idx], X.iloc[split_idx:],
        y.iloc[:split_idx], y.iloc[split_idx:], split_idx,
    )

def preparar_features(df, n_lags=5):
    df = df.copy()
    feature_cols = []
    for i in range(1, n_lags + 1):
        col_name = f"log_return_lag{i}"
        df[col_name] = df["log_return"].shift(i)
        feature_cols.append(col_name)
    df["target_next_day"] = df["spike"].shift(-1)
    df = df.dropna(subset=feature_cols + ["target_next_day"]).reset_index(drop=True)
    return df, feature_cols

trm_feat, feature_cols = preparar_features(trm_etiquetada, n_lags=5)
validar_conteo_modelo(len(trm_feat), 8354, "Conjunto de clasificación")
X_train, X_test, y_train, y_test, split_idx = split_temporal(
    trm_feat, feature_cols, target_col="target_next_day", pct_train=0.90
)
if trm_feat["fecha"].iloc[split_idx - 1] >= trm_feat["fecha"].iloc[split_idx]:
    raise AssertionError("Train y test no están separados cronológicamente.")
validar_conteo_modelo(len(X_train), 7518, "Partición train")
validar_conteo_modelo(len(X_test), 836, "Partición test")

spikes_train, spikes_test = int(y_train.sum()), int(y_test.sum())
print(f"Conjunto de clasificación: {len(trm_feat):,}")
print(
    f"Train: {len(X_train):,}; spikes={spikes_train}; "
    f"fechas {trm_feat['fecha'].iloc[0].date()}–{trm_feat['fecha'].iloc[split_idx-1].date()}"
)
print(
    f"Test: {len(X_test):,}; spikes={spikes_test}; "
    f"fechas {trm_feat['fecha'].iloc[split_idx].date()}–{trm_feat['fecha'].iloc[-1].date()}"
)


In [ ]:
# C14 — Métricas Fase 1, conteos y comparación con referencias
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)

if "X_train" not in globals() or "X_test" not in globals():
    raise RuntimeError("Ejecuta C13 primero: faltan las particiones.")
if "y_train" not in globals() or "y_test" not in globals():
    raise RuntimeError("Ejecuta C13 primero: faltan las etiquetas.")
if "feature_cols" not in globals() or "trm_feat" not in globals():
    raise RuntimeError("Ejecuta C13 primero: faltan features o trm_feat.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
if len(X_train) != len(y_train) or len(X_test) != len(y_test):
    raise AssertionError("X e y tienen tamaños incompatibles.")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)
modelo_logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE
    ),
)
modelo_logit.fit(X_train, y_train)
pred_logit = modelo_logit.predict(X_test)

metric_records = []
for nombre, pred in {
    "Mayoría": pred_dummy,
    "Regresión logística balanceada": pred_logit,
}.items():
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    row = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, pred),
        "recall_spike": recall_score(y_test, pred, zero_division=0),
        "precision_spike": precision_score(y_test, pred, zero_division=0),
        "f1_spike": f1_score(y_test, pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "n_test": len(y_test), "spikes_test": int(y_test.sum()),
    }
    metric_records.append(row)
    print(f"\n=== {nombre} ===")
    print(
        f"Accuracy={row['accuracy']:.4f}; Recall={row['recall_spike']:.4f}; "
        f"Precisión={row['precision_spike']:.4f}; F1={row['f1_spike']:.4f}"
    )
    print("Matriz [filas real 0/1, columnas predicho 0/1]:")
    print(cm)
    print(classification_report(y_test, pred, labels=[0, 1], zero_division=0))

metricas_fase1 = pd.DataFrame(metric_records)
metricas_fase1.to_csv(Path(RUN_DIR) / "metricas_fase1_test.csv", index=False)

columnas_exportacion = [
    "fecha", "trm", "log_return", "resid", "cond_sd", "spike",
    "target_next_day",
] + feature_cols
dataset_final = trm_feat[columnas_exportacion].copy()
dataset_final.to_csv(
    Path(RUN_DIR) / "trm_spikes_dataset_reconstruido.csv",
    index=False, date_format="%Y-%m-%d", float_format="%.10g",
)

COLUMNAS_NUMERICAS = ["trm", "log_return", "resid", "cond_sd", *feature_cols]
COLUMNAS_EXACTAS = ["spike", "target_next_day"]

def preparar_referencia(referencia, columna_fecha, nombre):
    requeridas = {columna_fecha, *COLUMNAS_EXACTAS, *COLUMNAS_NUMERICAS}
    faltantes = requeridas - set(referencia.columns)
    if faltantes:
        raise ValueError(f"{nombre}: faltan columnas {sorted(faltantes)}.")
    ref = referencia.copy().rename(columns={columna_fecha: "fecha"})
    ref["fecha"] = pd.to_datetime(ref["fecha"], errors="coerce").dt.normalize()
    if ref["fecha"].isna().any():
        raise ValueError(f"{nombre}: hay fechas inválidas.")
    return ref

def comparar_csv_principal(referencia):
    nombre = "trm_spikes_dataset.csv"
    ref = preparar_referencia(referencia, "fecha", nombre)
    actual = dataset_final.copy()
    actual["fecha"] = pd.to_datetime(actual["fecha"], errors="raise").dt.normalize()
    if not pd.DatetimeIndex(actual["fecha"]).equals(pd.DatetimeIndex(ref["fecha"])):
        raise AssertionError(f"{nombre}: las fechas no coinciden exactamente.")
    for col in COLUMNAS_EXACTAS:
        a = pd.to_numeric(actual[col], errors="coerce")
        b = pd.to_numeric(ref[col], errors="coerce")
        if a.isna().any() or b.isna().any() or not np.array_equal(a, b):
            raise AssertionError(f"{nombre}: difiere la columna exacta {col}.")
    for col in COLUMNAS_NUMERICAS:
        a = pd.to_numeric(actual[col], errors="coerce").to_numpy(dtype=float)
        b = pd.to_numeric(ref[col], errors="coerce").to_numpy(dtype=float)
        if np.isnan(a).any() or np.isnan(b).any():
            raise ValueError(f"{nombre}: valores numéricos inválidos en {col}.")
        max_diff = float(np.max(np.abs(a - b)))
        print(f"{nombre}: diferencia absoluta máxima {col}: {max_diff:.12g}")
        if max_diff > 1e-6:
            warnings.warn(
                f"{nombre}: diferencia de {col} supera 1e-6 ({max_diff:.12g}).",
                RuntimeWarning,
            )
    print(f"{nombre}: fechas, spike y target_next_day coinciden exactamente.")

def comparar_snapshot_informativo(referencia, columna_fecha, nombre):
    try:
        ref = preparar_referencia(referencia, columna_fecha, nombre)
    except Exception as exc:
        warnings.warn(f"No se pudo comparar {nombre}: {exc}", RuntimeWarning)
        return
    actual = dataset_final.copy()
    actual["fecha"] = pd.to_datetime(actual["fecha"], errors="coerce").dt.normalize()
    comunes = actual.merge(
        ref, on="fecha", how="inner", suffixes=("_nuevo", "_ref"),
        validate="one_to_one",
    )
    if len(comunes) != len(actual) or len(comunes) != len(ref):
        warnings.warn(
            f"{nombre}: distinto número de fechas; reconstrucción={len(actual)}, "
            f"snapshot={len(ref)}, comunes={len(comunes)}.",
            RuntimeWarning,
        )
    for col in COLUMNAS_EXACTAS:
        a = pd.to_numeric(comunes[f"{col}_nuevo"], errors="coerce")
        b = pd.to_numeric(comunes[f"{col}_ref"], errors="coerce")
        iguales = (
            not a.isna().any() and not b.isna().any()
            and np.array_equal(a.to_numpy(), b.to_numpy())
        )
        print(f"{nombre}: {col} coincide en fechas comunes={iguales}")
        if not iguales:
            warnings.warn(f"{nombre}: {col} difiere; referencia informativa.", RuntimeWarning)
    for col in COLUMNAS_NUMERICAS:
        a = pd.to_numeric(comunes[f"{col}_nuevo"], errors="coerce").to_numpy(dtype=float)
        b = pd.to_numeric(comunes[f"{col}_ref"], errors="coerce").to_numpy(dtype=float)
        max_diff = float(np.nanmax(np.abs(a - b))) if len(comunes) else float("nan")
        print(f"{nombre}: diferencia absoluta máxima {col}: {max_diff:.12g}")
        if np.isfinite(max_diff) and max_diff > 1e-6:
            warnings.warn(
                f"{nombre}: diferencia de {col} supera 1e-6; referencia informativa.",
                RuntimeWarning,
            )

comparar_csv_principal(trm_reference)
comparar_snapshot_informativo(
    trm_aprobada_ref, "date", "snapshot procesado trm_aprobada_fase1"
)

conteos_fase1 = pd.DataFrame([
    {"etapa": "Filas disponibles en snapshot crudo", "filas": filas_snapshot_crudo},
    {"etapa": "TRM válida", "filas": len(trm_valida)},
    {"etapa": "Retornos", "filas": len(trm_retornos)},
    {"etapa": "Serie etiquetada GARCH", "filas": len(trm_etiquetada)},
    {"etapa": "Clasificación con rezagos y objetivo", "filas": len(trm_feat)},
])
print("\nConteos de Fase 1:")
print(conteos_fase1.to_string(index=False))
conteos_fase1.to_csv(Path(RUN_DIR) / "conteos_fase1.csv", index=False)
print(f"\nMétricas y reconstrucción guardadas en: {RUN_DIR}")


In [ ]:
# C15 - Retornos de mercado desde snapshots
from pathlib import Path
import numpy as np
import pandas as pd

if "market" not in globals() or not isinstance(market, pd.DataFrame):
    raise RuntimeError("Ejecuta C04 primero: falta snapshot de mercado.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
close_cols = {
    "brent": "brent_close", "wti": "wti_close", "dxy": "dxy_close",
    "mxn": "mxn_close", "clp": "clp_close", "pen": "pen_close",
}
if not {"date", *close_cols.values()}.issubset(market.columns):
    raise ValueError("Faltan fecha o cierres de activos en el snapshot.")
market_snapshot = market.copy()
market_snapshot["date"] = pd.to_datetime(market_snapshot["date"], errors="coerce").dt.normalize()
if market_snapshot["date"].isna().any() or market_snapshot["date"].duplicated().any():
    raise ValueError("Fechas invalidas o duplicadas en snapshot de mercado.")
asset_returns = []
for asset, close_col in close_cols.items():
    s = market_snapshot[["date", close_col]].copy()
    s[close_col] = pd.to_numeric(s[close_col], errors="coerce")
    s = s.dropna().sort_values("date")
    ret_col = f"{asset}_return"
    s[ret_col] = s[close_col].pct_change(fill_method=None)
    s = s.replace([np.inf, -np.inf], np.nan).dropna(subset=[ret_col])
    asset_returns.append(s[["date", ret_col]])
market_returns = asset_returns[0]
for s in asset_returns[1:]:
    market_returns = market_returns.merge(s, on="date", how="outer", validate="one_to_one")
market_returns = market_returns.sort_values("date").reset_index(drop=True)
market_returns.to_csv(Path(RUN_DIR) / "retornos_mercado_reconstruidos.csv", index=False)
print(f"Se unieron seis series por fecha: {len(market_returns):,} filas.")

### Alineacion temporal y validacion cruzada de Fase 2

Cada serie de mercado se transforma a retornos por activo, se une por fecha y se alinea al origen con \`merge_asof\` backward y tolerancia de cinco dias. EMBI usa la misma direccion y tolerancia. BanRep usa backward sin tolerancia porque cada tasa sigue vigente hasta el cambio siguiente. El origen de prediccion es la fecha TRM previa; no se usan observaciones posteriores al origen.

La alineacion es determinista y ocurre antes de la validacion cruzada; no hay un \`TemporalAsOfAligner\` dentro del Pipeline. La imputacion y el escalado quedan dentro del Pipeline y se ajustan en cada particion de entrenamiento de la CV.

In [ ]:
# C16 - Panel Fase 2, alineacion y faltantes
from pathlib import Path
import numpy as np
import pandas as pd

for name in ("trm_feat", "market_returns", "banrep", "embi"):
    if name not in globals():
        raise RuntimeError(f"Falta {name}; ejecuta las celdas anteriores.")
if "verificar_conteo_fase2" not in globals():
    raise RuntimeError("C09 debe definir verificar_conteo_fase2.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")

base = trm_feat[["fecha", "log_return", "spike"]].copy().rename(columns={"fecha": "date"}).sort_values("date")
base["date"] = pd.to_datetime(base["date"], errors="raise").dt.normalize()
base["origin_date"] = base["date"].shift(1)
base["trm_return_lag1"] = pd.to_numeric(base["log_return"], errors="coerce").shift(1)
base = base.dropna(subset=["origin_date", "trm_return_lag1"]).reset_index(drop=True)
base["spike"] = base["spike"].astype(int)

market_src = market_returns.rename(columns={"date": "market_date"}).sort_values("market_date")
market_src["market_date"] = pd.to_datetime(market_src["market_date"], errors="raise").dt.normalize()
policy_src = banrep[["fecha_efectiva", "tasa_intervencion"]].rename(columns={"fecha_efectiva": "policy_date"}).sort_values("policy_date")
policy_src["policy_date"] = pd.to_datetime(policy_src["policy_date"], errors="raise").dt.normalize()
embi_src = embi[["date", "embi_colombia"]].rename(columns={"date": "embi_date"}).sort_values("embi_date")
embi_src["embi_date"] = pd.to_datetime(embi_src["embi_date"], errors="raise").dt.normalize()

left = base.copy()
left["_row_order"] = np.arange(len(left))
left = left.sort_values("origin_date")
panel_asof = pd.merge_asof(
    left, market_src, left_on="origin_date", right_on="market_date",
    direction="backward", allow_exact_matches=True, tolerance=pd.Timedelta(days=5),
)
panel_asof = pd.merge_asof(
    panel_asof.sort_values("origin_date"), policy_src,
    left_on="origin_date", right_on="policy_date",
    direction="backward", allow_exact_matches=True, tolerance=None,
)
panel_asof = pd.merge_asof(
    panel_asof.sort_values("origin_date"), embi_src,
    left_on="origin_date", right_on="embi_date",
    direction="backward", allow_exact_matches=True, tolerance=pd.Timedelta(days=5),
).sort_values("_row_order").reset_index(drop=True)
for col in ("market_date", "policy_date", "embi_date"):
    if (panel_asof[col].notna() & (panel_asof[col] > panel_asof["origin_date"])).any():
        raise AssertionError(f"Look-ahead detectado en {col}.")

missing_cols = {
    "brent_return": "Brent", "wti_return": "WTI", "dxy_return": "DXY",
    "mxn_return": "MXN", "clp_return": "CLP", "pen_return": "PEN",
    "embi_colombia": "EMBI",
}
denominator = len(panel_asof)
missing_table = pd.DataFrame([
    {"variable": label, "columna": col, "faltantes": int(panel_asof[col].isna().sum()),
     "porcentaje": 100 * panel_asof[col].isna().mean(), "denominador": denominator}
    for col, label in missing_cols.items()
])
print(f"Faltantes antes del filtro EMBI; denominador={denominator:,}.")
print(missing_table.to_string(index=False, formatters={"porcentaje": "{:.2f}%".format}))
missing_table.to_csv(Path(RUN_DIR) / "faltantes_antes_filtro_embi.csv", index=False)

panel_fase2 = panel_asof.dropna(subset=["origin_date", "spike", "embi_colombia"]).copy()
panel_fase2 = panel_fase2.sort_values("date").reset_index(drop=True)
panel_fase2["spike"] = panel_fase2["spike"].astype(int)
market_feature_cols = ["brent_return", "wti_return", "dxy_return", "mxn_return", "clp_return", "pen_return"]
feature_cols_fase2 = ["trm_return_lag1", *market_feature_cols, "tasa_intervencion", "embi_colombia"]
# Faltantes de mercado permanecen para imputacion dentro de cada fold.
if panel_fase2["tasa_intervencion"].isna().any() or panel_fase2["embi_colombia"].isna().any():
    raise AssertionError("Quedaron faltantes de BanRep o EMBI en el panel final.")
verificar_conteo_fase2(len(panel_fase2), "panel_filas")
verificar_conteo_fase2(int(panel_fase2["spike"].sum()), "panel_spikes")
print(f"Panel final: {len(panel_fase2):,}; spikes={int(panel_fase2.spike.sum()):,}")
panel_fase2.to_csv(Path(RUN_DIR) / "panel_fase2.csv", index=False, date_format="%Y-%m-%d")

In [ ]:
# C17 - Particion temporal, Pipeline y validacion temporal
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

if "panel_fase2" not in globals() or "feature_cols_fase2" not in globals():
    raise RuntimeError("Ejecuta C16 primero.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
panel_fase2 = panel_fase2.sort_values("date").reset_index(drop=True)
cut = int(len(panel_fase2) * .90)
train_fase2, test_fase2 = panel_fase2.iloc[:cut].copy(), panel_fase2.iloc[cut:].copy()
X_train_fase2, y_train_fase2 = train_fase2[feature_cols_fase2], train_fase2["spike"].astype(int)
X_test_fase2, y_test_fase2 = test_fase2[feature_cols_fase2], test_fase2["spike"].astype(int)
for actual, key in [(len(train_fase2), "train_filas"), (len(test_fase2), "test_filas"),
                    (int(y_test_fase2.sum()), "test_spikes")]:
    verificar_conteo_fase2(actual, key)
if train_fase2.date.max() >= test_fase2.date.min():
    raise AssertionError("La particion no es cronologica.")

def pipeline_svm(kernel):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel=kernel, class_weight="balanced", probability=True,
                    random_state=RANDOM_STATE, cache_size=1000)),
    ])

cv_splits_fase2 = list(TimeSeriesSplit(n_splits=5).split(X_train_fase2))
for fold, (itr, iva) in enumerate(cv_splits_fase2, 1):
    print(f"Fold {fold}: spikes train={int(y_train_fase2.iloc[itr].sum())}; "
          f"validacion={int(y_train_fase2.iloc[iva].sum())}")
scoring_fase2 = {
    "recall": "recall", "precision": "precision", "f1": "f1",
    "roc_auc": "roc_auc", "average_precision": "average_precision",
}
grids_fase2 = {
    "SVM lineal": (pipeline_svm("linear"), {"svc__C": [.01, .1, 1, 10, 100]}),
    "SVM RBF": (pipeline_svm("rbf"), {
        "svc__C": [.1, 1, 10, 100], "svc__gamma": ["scale", .001, .01, .1, 1],
    }),
}
modelos_fase2, cv_summary, cv_candidates = {}, [], []
for name, (estimator, param_grid) in grids_fase2.items():
    search = GridSearchCV(estimator, param_grid, scoring=scoring_fase2, refit="recall",
                          cv=cv_splits_fase2, n_jobs=-1, return_train_score=False,
                          error_score="raise").fit(X_train_fase2, y_train_fase2)
    modelos_fase2[name] = search.best_estimator_
    best = search.best_index_
    row = {"modelo": name, "best_params": json.dumps(search.best_params_, sort_keys=True)}
    for metric in scoring_fase2:
        row[f"cv_{metric}_mean"] = search.cv_results_[f"mean_test_{metric}"][best]
        row[f"cv_{metric}_std"] = search.cv_results_[f"std_test_{metric}"][best]
    cv_summary.append(row)
    for i, params in enumerate(search.cv_results_["params"]):
        cand = {"modelo": name, "params": json.dumps(params, sort_keys=True)}
        for metric in scoring_fase2:
            cand[f"cv_{metric}_mean"] = search.cv_results_[f"mean_test_{metric}"][i]
            cand[f"cv_{metric}_std"] = search.cv_results_[f"std_test_{metric}"][i]
        cv_candidates.append(cand)
    print(f"{name}: mejores parametros {search.best_params_}")
cv_summary_df, cv_candidates_df = pd.DataFrame(cv_summary), pd.DataFrame(cv_candidates)
cv_summary_df.to_csv(Path(RUN_DIR) / "resultados_cv_svm.csv", index=False)
cv_candidates_df.to_csv(Path(RUN_DIR) / "fase2_cv_candidatos.csv", index=False)
modelo_logit_fase2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)),
]).fit(X_train_fase2, y_train_fase2)
modelo_mayoria_fase2 = DummyClassifier(strategy="most_frequent").fit(X_train_fase2, y_train_fase2)

In [ ]:
# C18 - Evaluacion de test y metricas Fase 2
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)

if "modelos_fase2" not in globals() or "y_test_fase2" not in globals():
    raise RuntimeError("Ejecuta C17 primero.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
evaluadores = {
    "Mayoria": modelo_mayoria_fase2,
    "Regresion logistica": modelo_logit_fase2,
    **modelos_fase2,
}
resultados_fase2, detalle_modelos_fase2 = [], []
predicciones = pd.DataFrame({"fecha": test_fase2.date.to_numpy(), "y_real": y_test_fase2.to_numpy()})
for name, model in evaluadores.items():
    pred = model.predict(X_test_fase2)
    score = model.predict_proba(X_test_fase2)[:, 1]
    cm = confusion_matrix(y_test_fase2, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    resultados_fase2.append({
        "modelo": name, "accuracy": accuracy_score(y_test_fase2, pred),
        "recall_spike": recall_score(y_test_fase2, pred, zero_division=0),
        "precision_spike": precision_score(y_test_fase2, pred, zero_division=0),
        "f1_spike": f1_score(y_test_fase2, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test_fase2, score),
        "average_precision": average_precision_score(y_test_fase2, score),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "confusion_matrix": json.dumps(cm.tolist()),
        "n_test": len(y_test_fase2), "spikes_test": int(y_test_fase2.sum()),
    })
    detalle_modelos_fase2.append({"modelo": name, "pred": pred, "score": score, "cm": cm})
    slug = name.lower().replace(" ", "_")
    predicciones[f"{slug}_pred"], predicciones[f"{slug}_score"] = pred, score
    print(f"\n=== {name} ===\n{classification_report(y_test_fase2, pred, zero_division=0)}")
metricas_fase2 = pd.DataFrame(resultados_fase2)
metricas_fase2.to_csv(Path(RUN_DIR) / "resultados_modelos_segunda_entrega.csv", index=False)
predicciones.to_csv(Path(RUN_DIR) / "predicciones_test_fase2.csv", index=False, date_format="%Y-%m-%d")
print(metricas_fase2.to_string(index=False))

In [ ]:
# C19 - Figuras y comparacion con resultados de referencia
from pathlib import Path
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_auc_score, roc_curve

if "metricas_fase2" not in globals() or "cv_summary_df" not in globals():
    raise RuntimeError("Ejecuta C17 y C18 primero.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
FIGURES_FASE2_DIR = Path(RUN_DIR) / "figures_fase2"
FIGURES_FASE2_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (x, y) in zip(axes, [
    ("brent_return", "wti_return"), ("dxy_return", "embi_colombia"),
    ("tasa_intervencion", "embi_colombia"),
]):
    sample = panel_fase2[[x, y, "spike"]].dropna()
    if len(sample) > 2500:
        sample = sample.sample(2500, random_state=42)
    ax.scatter(sample[x], sample[y], c=np.where(sample.spike == 1, "#d62728", "#3977a8"), alpha=.45, s=20)
    ax.set(xlabel=x, ylabel=y, title=f"{x} vs {y}")
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_FASE2_DIR / "dispersion_predictores_exploratoria.png", dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for item in detalle_modelos_fase2:
    name, score = item["modelo"], item["score"]
    fpr, tpr, _ = roc_curve(y_test_fase2, score)
    precision, recall, _ = precision_recall_curve(y_test_fase2, score)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test_fase2, score):.3f})")
    axes[1].plot(recall, precision, label=name)
axes[0].plot([0, 1], [0, 1], "k--", alpha=.5)
axes[1].axhline(y_test_fase2.mean(), color="black", linestyle="--")
axes[0].set(xlabel="FPR", ylabel="Recall", title="ROC - test")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall - test")
for ax in axes:
    ax.grid(alpha=.25); ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_FASE2_DIR / "curvas_modelos.png", dpi=200)
plt.show()

fig, axes = plt.subplots(1, len(detalle_modelos_fase2), figsize=(4 * len(detalle_modelos_fase2), 4))
for ax, item in zip(axes, detalle_modelos_fase2):
    ConfusionMatrixDisplay(item["cm"], display_labels=["no-spike", "spike"]).plot(
        ax=ax, colorbar=False, values_format="d")
    ax.set_title(item["modelo"])
fig.tight_layout()
fig.savefig(FIGURES_FASE2_DIR / "matrices_confusion_test.png", dpi=200)
plt.show()

metricas_fase2.set_index("modelo")[
    ["recall_spike", "precision_spike", "f1_spike", "roc_auc", "average_precision"]
].plot(kind="bar", ylim=(0, 1), figsize=(11, 5), rot=0)
plt.tight_layout(); plt.savefig(FIGURES_FASE2_DIR / "comparacion_metricas.png", dpi=200); plt.show()
colors = np.where(test_fase2.spike.to_numpy() == 1, "#d62728", "#3977a8")
plt.figure(figsize=(12, 4.5))
plt.scatter(test_fase2.date, test_fase2.trm_return_lag1, c=colors, alpha=.75)
plt.axhline(0, color="black", linewidth=.8)
plt.title("Retorno rezagado TRM por clase observada en test")
plt.xlabel("Fecha objetivo"); plt.ylabel("Retorno TRM rezagado"); plt.grid(alpha=.2)
plt.tight_layout(); plt.savefig(FIGURES_FASE2_DIR / "puntos_retorno_trm_test.png", dpi=200); plt.show()

# Auditoria de reproducibilidad; no selecciona modelos ni umbrales.
ref_dir = Path(PROJECT_DIR) / "results" / "fase2"
metric_ref, cv_ref = ref_dir / "resultados_modelos_segunda_entrega.csv", ref_dir / "resultados_cv_svm.csv"
differences = []

def report_diff(kind, model, field, old, new, tol=1e-6):
    if pd.isna(old) or pd.isna(new):
        state, delta = "no comparable", np.nan
    else:
        delta = abs(float(old) - float(new))
        state = "coincide" if delta <= tol else "difiere"
        if delta > tol:
            warnings.warn(f"{kind} {model}/{field} difiere por {delta:.8g}.", RuntimeWarning)
    differences.append({
        "tipo": kind, "modelo": model, "campo": field, "referencia": old,
        "corrida": new, "diferencia_absoluta": delta, "tolerancia": tol, "estado": state,
    })
    print(f"{kind} | {model} | {field}: {state}; diferencia={delta}")

if metric_ref.is_file():
    old_df, new_df = pd.read_csv(metric_ref).set_index("modelo"), metricas_fase2.set_index("modelo")
    name_map = {"Mayoria": "Mayor\u00eda", "Regresion logistica": "Regresi\u00f3n log\u00edstica"}
    for model in new_df.index:
        old_name = name_map.get(model, model)
        if old_name not in old_df.index:
            warnings.warn(f"Falta referencia de metricas para {old_name}.", RuntimeWarning)
            continue
        for field in ("accuracy", "recall_spike", "precision_spike", "f1_spike", "roc_auc", "average_precision"):
            report_diff("metrica", old_name, field, old_df.loc[old_name, field], new_df.loc[model, field])
else:
    warnings.warn(f"No existe {metric_ref}.", RuntimeWarning)

def parse_params(value):
    try:
        return ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return value

if cv_ref.is_file():
    old_cv, new_cv = pd.read_csv(cv_ref).set_index("modelo"), cv_summary_df.set_index("modelo")
    for model in new_cv.index:
        if model not in old_cv.index:
            warnings.warn(f"Falta referencia CV para {model}.", RuntimeWarning)
            continue
        old_params, new_params = parse_params(old_cv.loc[model, "best_params"]), parse_params(new_cv.loc[model, "best_params"])
        same = old_params == new_params
        differences.append({
            "tipo": "hiperparametros", "modelo": model, "campo": "best_params",
            "referencia": str(old_params), "corrida": str(new_params),
            "diferencia_absoluta": 0 if same else np.nan, "tolerancia": 0,
            "estado": "coincide" if same else "difiere",
        })
        print(f"hiperparametros | {model}: {'coincide' if same else 'difiere'}")
        for field in ("cv_recall_mean", "cv_recall_std"):
            report_diff("CV", model, field, old_cv.loc[model, field], new_cv.loc[model, field])
else:
    warnings.warn(f"No existe {cv_ref}.", RuntimeWarning)

comparison_path = Path(RUN_DIR) / "comparacion_referencias_fase2.csv"
pd.DataFrame(differences).to_csv(comparison_path, index=False)
print(f"Figuras: {FIGURES_FASE2_DIR}")
print(f"Comparacion con resultados/fase2: {comparison_path}")

In [ ]:
# C20 — Protocolo de selección Fase 3
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

if "MODO_RAPIDO" not in globals():
    raise RuntimeError("C02 debe definir MODO_RAPIDO antes de C09 y C20.")
if "RUN_DIR" not in globals() or not Path(RUN_DIR).is_dir():
    raise RuntimeError("Ejecuta C02 primero: RUN_DIR no existe.")
if "X_train_fase2" not in globals() or "y_train_fase2" not in globals():
    raise RuntimeError("Ejecuta C17 primero: faltan los datos de desarrollo.")

RANDOM_STATE = 42
MIN_SPIKES_BLOQUE = 10
UMBRAL_GRID_FASE3 = np.unique(np.round(np.concatenate([
    np.arange(0.00, 0.2001, 0.005),
    np.arange(0.20, 1.0001, 0.02),
]), 3))
if len(UMBRAL_GRID_FASE3) != 81:
    raise AssertionError(f"La grilla mixta debe tener 81 umbrales, tiene {len(UMBRAL_GRID_FASE3)}.")
print("Grilla mixta de 81 umbrales: mayor resolucion cerca de la prevalencia; "
      "la calibracion sigmoide puede concentrar alli scores de los SVM.")

if MODO_RAPIDO:
    N_OUTER_FASE3, N_INNER_FASE3, N_CAL_FASE3 = 2, 2, 2
    GRILLAS_FASE3 = {
        "Mayoría": [{}],
        "Regresión logística": [{"C": 1.0}],
        "SVM lineal": [{"C": 1.0}],
        "SVM RBF": [{"C": 1.0, "gamma": "scale"}],
    }
    print("MODO_RAPIDO=True: grillas y particiones reducidas; resultado diagnóstico.")
else:
    N_OUTER_FASE3, N_INNER_FASE3, N_CAL_FASE3 = 4, 3, 3
    GRILLAS_FASE3 = {
        "Mayoría": [{}],
        "Regresión logística": [{"C": c} for c in (0.1, 1.0, 10.0)],
        "SVM lineal": [{"C": c} for c in (0.1, 1.0, 10.0)],
        "SVM RBF": [
            {"C": c, "gamma": gamma}
            for c in (0.1, 1.0, 10.0)
            for gamma in ("scale", 0.01, 0.1)
        ],
    }
    print("MODO_RAPIDO=False: configuración completa.")

print(
    f"CV externa={N_OUTER_FASE3}; CV interna={N_INNER_FASE3}; "
    f"calibración temporal={N_CAL_FASE3}; umbrales={len(UMBRAL_GRID_FASE3)}."
)
print(
    "El test congelado no participa en selección, ajuste ni elección del umbral; "
    "se usa únicamente en C29."
)


In [ ]:
# C21 — Particiones externas cronológicas comunes; solo usa el tramo de desarrollo
from pathlib import Path
import warnings
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

if "X_train_fase2" not in globals() or "y_train_fase2" not in globals():
    raise RuntimeError("Ejecuta C17 primero.")
if len(X_train_fase2) != len(y_train_fase2):
    raise ValueError("X_train_fase2 e y_train_fase2 tienen longitudes distintas.")
if y_train_fase2.nunique() < 2:
    raise ValueError("El tramo de desarrollo no contiene ambas clases.")
if "N_OUTER_FASE3" not in globals():
    raise RuntimeError("Ejecuta C20 primero.")

cv_externo_fase3 = list(
    TimeSeriesSplit(n_splits=N_OUTER_FASE3).split(X_train_fase2)
)
registros_particiones_fase3 = []

for fold, (idx_train, idx_valid) in enumerate(cv_externo_fase3, start=1):
    ytr = y_train_fase2.iloc[idx_train]
    yva = y_train_fase2.iloc[idx_valid]
    spikes_valid = int(yva.sum())
    estable_eventos = spikes_valid >= MIN_SPIKES_BLOQUE
    ambas_clases_train = ytr.nunique() == 2

    if not estable_eventos:
        warnings.warn(
            f"Externa {fold}: {spikes_valid} spikes de validación; "
            f"menos del mínimo {MIN_SPIKES_BLOQUE}. Se reportará como inestable.",
            RuntimeWarning,
        )
    if not ambas_clases_train:
        warnings.warn(
            f"Externa {fold}: entrenamiento sin ambas clases; "
            "no se usará para ajustar modelos.",
            RuntimeWarning,
        )

    registros_particiones_fase3.append({
        "particion": fold,
        "n_train": len(idx_train),
        "spikes_train": int(ytr.sum()),
        "n_validacion": len(idx_valid),
        "spikes_validacion": spikes_valid,
        "entrenamiento_dos_clases": ambas_clases_train,
        "validacion_minimo_10_spikes": estable_eventos,
        "fecha_train_fin": str(train_fase2.iloc[idx_train]["date"].max().date()),
        "fecha_valid_inicio": str(train_fase2.iloc[idx_valid]["date"].min().date()),
        "fecha_valid_fin": str(train_fase2.iloc[idx_valid]["date"].max().date()),
    })

particiones_externas_fase3 = pd.DataFrame(registros_particiones_fase3)
print(particiones_externas_fase3.to_string(index=False))
particiones_externas_fase3.to_csv(
    Path(RUN_DIR) / "particiones_externas_fase3.csv", index=False
)


In [ ]:
# C22 — Seleccion temporal anidada y tabla completa de umbrales
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

for name in ("X_train_fase2", "y_train_fase2", "GRILLAS_FASE3", "UMBRAL_GRID_FASE3", "RUN_DIR"):
    if name not in globals():
        raise RuntimeError(f"Falta {name}; ejecuta C02, C17 y C20 primero.")
if len(X_train_fase2) != len(y_train_fase2) or len(UMBRAL_GRID_FASE3) != 81:
    raise ValueError("Datos desalineados o grilla de umbrales distinta de 81 valores.")

import joblib
from sklearn.metrics import confusion_matrix

FASE3_DIR = Path(RUN_DIR) / "fase3"
CHECKPOINT_DIR_FASE3 = FASE3_DIR / "checkpoints"
CHECKPOINT_DIR_FASE3.mkdir(parents=True, exist_ok=True)
TOL_RECALL_FASE3, TOL_FPR_FASE3 = 0.60, 0.50
MAX_ALERT_RATE_FASE3 = 0.95
CONFIG_HASH_INPUT = {
    "GRILLAS_FASE3": GRILLAS_FASE3, "MODO_RAPIDO": bool(MODO_RAPIDO),
    "N_OUTER_FASE3": N_OUTER_FASE3, "N_INNER_FASE3": N_INNER_FASE3,
    "N_CAL_FASE3": N_CAL_FASE3,
    "UMBRAL_GRID_FASE3": [float(x) for x in UMBRAL_GRID_FASE3],
    "MIN_SPIKES_BLOQUE": MIN_SPIKES_BLOQUE,
    "TOL_RECALL_FASE3": TOL_RECALL_FASE3, "TOL_FPR_FASE3": TOL_FPR_FASE3,
    "MAX_ALERT_RATE_FASE3": MAX_ALERT_RATE_FASE3,
}
CONFIG_HASH_FASE3 = hashlib.sha256(
    json.dumps(CONFIG_HASH_INPUT, sort_keys=True, default=str).encode("utf-8")
).hexdigest()

def _hash_xy_fase3(X, y):
    h = hashlib.sha256()
    h.update(pd.util.hash_pandas_object(X, index=True).values.tobytes())
    h.update(pd.util.hash_pandas_object(pd.Series(y, index=X.index), index=True).values.tobytes())
    return h.hexdigest()

def _atomic_csv_fase3(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def _pipeline_base_fase3(model_name, params):
    prep = [
        ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("scaler", StandardScaler()),
    ]
    if model_name == "Regresión logística":
        return Pipeline(prep + [("model", LogisticRegression(
            C=float(params["C"]), class_weight="balanced", max_iter=3000,
            random_state=RANDOM_STATE,
        ))])
    kernel = "linear" if model_name == "SVM lineal" else "rbf"
    svc_params = {"C": float(params["C"]), "kernel": kernel,
                  "class_weight": "balanced", "probability": False,
                  "random_state": RANDOM_STATE, "cache_size": 1000}
    if kernel == "rbf":
        svc_params["gamma"] = params["gamma"]
    return Pipeline(prep + [("model", SVC(**svc_params))])

def _fit_score_fase3(model_name, params, Xtr, ytr, Xev, n_cal):
    ytr = pd.Series(ytr, index=Xtr.index).astype(int)
    if ytr.nunique() < 2:
        raise ValueError("El bloque de entrenamiento no contiene ambas clases.")
    if model_name == "Mayoría":
        estimator = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
        return estimator, estimator.predict_proba(Xev)[:, 1]
    estimator = _pipeline_base_fase3(model_name, params)
    if model_name == "Regresión logística":
        estimator.fit(Xtr, ytr)
        return estimator, estimator.predict_proba(Xev)[:, 1]

    # Calibración sigmoide propia sobre scores OOF de cortes temporales expansivos:
    # cada score de calibración procede de un ajuste que solo vio fechas anteriores.
    split_count = min(int(n_cal), max(0, len(Xtr) // 20))
    if split_count < 2:
        raise ValueError("No hay longitud suficiente para dos cortes temporales de calibración.")
    oof_scores, oof_labels = [], []
    for itr, iva in TimeSeriesSplit(n_splits=split_count).split(Xtr):
        y_fit = ytr.iloc[itr]
        if y_fit.nunique() < 2:
            continue
        fold_estimator = _pipeline_base_fase3(model_name, params).fit(Xtr.iloc[itr], y_fit)
        fold_scores = fold_estimator.decision_function(Xtr.iloc[iva])
        oof_scores.extend(np.asarray(fold_scores, dtype=float).reshape(-1).tolist())
        oof_labels.extend(ytr.iloc[iva].tolist())
    if len(oof_scores) < 20 or len(set(oof_labels)) < 2:
        raise ValueError("Scores OOF insuficientes o sin ambas clases para calibrar el SVM.")
    calibrator = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000,
                                    random_state=RANDOM_STATE)
    calibrator.fit(np.asarray(oof_scores).reshape(-1, 1), np.asarray(oof_labels))
    estimator.fit(Xtr, ytr)
    scores = estimator.decision_function(Xev)
    probabilities = calibrator.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]
    return {"base": estimator, "calibrator": calibrator}, probabilities

def _measure_threshold_fase3(y, scores, threshold, model_name, params_json, fold, trivial=False):
    y = np.asarray(y, dtype=int)
    pred = np.ones(len(y), dtype=int) if trivial else (np.asarray(scores) >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if tp + fn else np.nan
    precision = tp / (tp + fp) if tp + fp else 0.0
    fpr = fp / (fp + tn) if fp + tn else np.nan
    f2 = (5 * precision * recall / (4 * precision + recall)) if (4 * precision + recall) else 0.0
    return {
        "particion_interna": fold, "modelo": model_name, "parametros": params_json,
        "umbral": np.nan if trivial else float(threshold), "n_validacion": len(y),
        "spikes_validacion": int(y.sum()), "alert_rate": float(pred.mean()),
        "recall": float(recall), "precision": float(precision), "fpr": float(fpr),
        "fp": int(fp), "fn": int(fn), "tn": int(tn), "tp": int(tp),
        "f2": float(f2), "estado": "referencia_trivial" if trivial else "candidato",
    }

def _run_inner_cv_fase3(X, y, context, n_splits):
    y = pd.Series(y, index=X.index).astype(int)
    if len(X) != len(y):
        raise ValueError(f"{context}: X e y desalineados.")
    folds = list(TimeSeriesSplit(n_splits=n_splits).split(X))
    records = []
    for fold, (itr, iva) in enumerate(folds, 1):
        fold_hash = _hash_xy_fase3(X.iloc[np.r_[itr, iva]], y.iloc[np.r_[itr, iva]])
        stem = f"{context}_inner_{fold}"
        cache_path = CHECKPOINT_DIR_FASE3 / f"{stem}.csv"
        meta_path = CHECKPOINT_DIR_FASE3 / f"{stem}.json"
        if cache_path.is_file() and meta_path.is_file():
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            if meta.get("config_hash") != CONFIG_HASH_FASE3 or meta.get("input_hash") != fold_hash:
                raise RuntimeError(f"Checkpoint incompatible: {stem}; cambia run_id o limpia solo esa corrida.")
            fold_records = pd.read_csv(cache_path).to_dict("records")
            records.extend(fold_records)
            continue
        ytr, yva = y.iloc[itr], y.iloc[iva]
        if int(yva.sum()) < MIN_SPIKES_BLOQUE or ytr.nunique() < 2 or yva.nunique() < 2:
            fold_records = [{"particion_interna": fold, "modelo": "", "parametros": "",
                             "umbral": np.nan, "n_validacion": len(iva),
                             "spikes_validacion": int(yva.sum()), "alert_rate": np.nan,
                             "recall": np.nan, "precision": np.nan, "fpr": np.nan,
                             "fp": np.nan, "fn": np.nan, "tn": np.nan, "tp": np.nan,
                             "f2": np.nan, "estado": "no_evaluable"}]
        else:
            fold_records = [_measure_threshold_fase3(yva, None, None, "Referencia trivial", "{}", fold, True)]
            # Para seleccion, la referencia trivial se mantiene separada y no es candidata.
            fold_records[0]["estado"] = "referencia_trivial"
            for model_name, configurations in GRILLAS_FASE3.items():
                for params in configurations:
                    params_json = json.dumps(params, sort_keys=True)
                    try:
                        _, scores = _fit_score_fase3(
                            model_name, params, X.iloc[itr], ytr, X.iloc[iva], N_CAL_FASE3
                        )
                    except Exception as exc:
                        warnings.warn(f"{context} fold {fold}, {model_name} {params}: {exc}", RuntimeWarning)
                        continue
                    fold_records.extend(
                        _measure_threshold_fase3(yva, scores, threshold, model_name,
                                                 params_json, fold)
                        for threshold in UMBRAL_GRID_FASE3
                    )
        _atomic_csv_fase3(pd.DataFrame(fold_records), cache_path)
        meta_path.write_text(json.dumps({"config_hash": CONFIG_HASH_FASE3,
                                         "input_hash": fold_hash}, indent=2), encoding="utf-8")
        records.extend(fold_records)
    raw = pd.DataFrame(records)
    valid = raw[(raw["estado"] == "candidato")].copy() if not raw.empty else raw
    if valid.empty:
        return raw, pd.DataFrame()
    keys = ["modelo", "parametros", "umbral"]
    agg = valid.groupby(keys, dropna=False).agg(
        f2_mean=("f2", "mean"), f2_std=("f2", "std"),
        recall_mean=("recall", "mean"), recall_std=("recall", "std"),
        precision_mean=("precision", "mean"), fpr_mean=("fpr", "mean"),
        fp_mean=("fp", "mean"), fn_mean=("fn", "mean"),
        alert_rate=("alert_rate", "mean"), alert_rate_max=("alert_rate", "max"),
        n_folds=("particion_interna", "nunique"),
    ).reset_index()
    tol_by_fold = valid.assign(tolerancia=(valid.recall >= TOL_RECALL_FASE3) &
                                (valid.fpr <= TOL_FPR_FASE3)).groupby(keys).tolerancia.sum()
    agg["tolerancia_folds"] = [int(tol_by_fold.get(tuple(row[k] for k in keys), 0)) for _, row in agg.iterrows()]
    agg["estable"] = agg["tolerancia_folds"] > agg["n_folds"] / 2
    # Se excluye un umbral si supera 95% de alertas en cualquier fold que lo evaluó.
    agg["elegible_alert_rate"] = agg["alert_rate_max"] <= MAX_ALERT_RATE_FASE3
    return raw, agg

def _choose_fase3(agg, model_name=None):
    if agg.empty:
        return None
    pool = agg if model_name is None else agg[agg.modelo == model_name]
    pool = pool[pool.elegible_alert_rate].copy()
    if pool.empty:
        return None
    stable = pool[pool.estable]
    if not stable.empty:
        pool = stable
        status = "cumple_tolerancia_en_mayoria_interna"
    else:
        status = "no cumple la tolerancia"
    complexity = {"Mayoría": 0, "Regresión logística": 1, "SVM lineal": 2, "SVM RBF": 3}
    pool["_complexity"] = pool.modelo.map(complexity).fillna(9)
    best = pool.sort_values(["f2_mean", "f2_std", "_complexity", "umbral"],
                            ascending=[False, True, True, False], kind="mergesort").iloc[0].drop(labels="_complexity")
    return {**best.to_dict(), "estado_tolerancia": status}

run_hash = _hash_xy_fase3(X_train_fase2, y_train_fase2)
raw_interna_fase3, tabla_umbrales_internos_fase3 = _run_inner_cv_fase3(
    X_train_fase2, y_train_fase2, "global", N_INNER_FASE3
)
if tabla_umbrales_internos_fase3.empty:
    raise RuntimeError("Ningún bloque interno global fue evaluable; no se puede seleccionar.")
selecciones_por_modelo_fase3 = []
for model_name in GRILLAS_FASE3:
    chosen = _choose_fase3(tabla_umbrales_internos_fase3, model_name)
    if chosen is not None:
        chosen["modelo"] = model_name
        selecciones_por_modelo_fase3.append(chosen)
seleccion_candidatos_fase3 = pd.DataFrame(selecciones_por_modelo_fase3)
seleccion_global_fase3 = _choose_fase3(tabla_umbrales_internos_fase3)
if seleccion_global_fase3 is None:
    raise RuntimeError("No hay umbral elegible (alert_rate <= 95%) para la selección global.")
seleccion_global_fase3["run_id"] = RUN_ID
seleccion_global_fase3["input_sha256"] = run_hash
seleccion_global_fase3["config_sha256"] = CONFIG_HASH_FASE3
tabla_umbrales_internos_fase3.to_csv(FASE3_DIR / "tabla_umbrales_internos.csv", index=False)
raw_interna_fase3.to_csv(FASE3_DIR / "tabla_umbrales_por_fold_internos.csv", index=False)
seleccion_candidatos_fase3.to_csv(FASE3_DIR / "seleccion_por_modelo_interna.csv", index=False)
pd.DataFrame([seleccion_global_fase3]).to_csv(FASE3_DIR / "seleccion_global_desarrollo.csv", index=False)
print("Tabla completa de umbrales guardada; referencia trivial excluida de la selección.")
trivial_interno = raw_interna_fase3[raw_interna_fase3["estado"] == "referencia_trivial"]
print("Referencia trivial interna (alertar siempre):")
print(trivial_interno[["particion_interna", "n_validacion", "spikes_validacion", "alert_rate", "recall", "precision", "fpr", "f2"]].to_string(index=False))
print("Seleccion desarrollo:", {k: seleccion_global_fase3[k] for k in
      ("modelo", "parametros", "umbral", "f2_mean", "recall_mean", "precision_mean",
       "fpr_mean", "alert_rate", "estado_tolerancia")})
print(f"Hash de datos={run_hash}; hash de configuración={CONFIG_HASH_FASE3}")


In [ ]:
# C23 — Evaluación externa anidada, checkpoints y resumen descriptivo
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

if "cv_externo_fase3" not in globals() or "_run_inner_cv_fase3" not in globals():
    raise RuntimeError("Ejecuta C21 y C22 primero.")
if not cv_externo_fase3:
    raise RuntimeError("No hay particiones externas configuradas.")

external_rows, prediction_frames = [], []
evaluable_outer = 0
for outer_id, (idx_train, idx_valid) in enumerate(cv_externo_fase3, 1):
    Xtr, ytr = X_train_fase2.iloc[idx_train], y_train_fase2.iloc[idx_train].astype(int)
    Xva, yva = X_train_fase2.iloc[idx_valid], y_train_fase2.iloc[idx_valid].astype(int)
    outer_hash = _hash_xy_fase3(
        X_train_fase2.iloc[np.r_[idx_train, idx_valid]],
        y_train_fase2.iloc[np.r_[idx_train, idx_valid]],
    )
    outer_key = f"outer_{outer_id}"
    metrics_path = CHECKPOINT_DIR_FASE3 / f"{outer_key}_metrics.csv"
    preds_path = CHECKPOINT_DIR_FASE3 / f"{outer_key}_predicciones.csv"
    meta_path = CHECKPOINT_DIR_FASE3 / f"{outer_key}.json"
    if metrics_path.is_file() and preds_path.is_file() and meta_path.is_file():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("config_hash") != CONFIG_HASH_FASE3 or meta.get("input_hash") != outer_hash:
            raise RuntimeError(f"Checkpoint externo incompatible: {outer_key}.")
        fold_metrics = pd.read_csv(metrics_path)
        fold_preds = pd.read_csv(preds_path)
        if not fold_metrics.empty and (fold_metrics.estado == "evaluable").any():
            evaluable_outer += 1
        external_rows.extend(fold_metrics.to_dict("records"))
        prediction_frames.append(fold_preds)
        print(f"Externa {outer_id}: reanudada desde checkpoint.")
        continue

    if int(yva.sum()) < MIN_SPIKES_BLOQUE or yva.nunique() < 2:
        warnings.warn(f"Externa {outer_id}: bloque externo no evaluable; spikes={int(yva.sum())}.", RuntimeWarning)
        fold_metrics = pd.DataFrame([{"particion_externa": outer_id, "modelo": "",
                                     "estado": "no_evaluable", "n_validacion": len(yva),
                                     "spikes_validacion": int(yva.sum())}])
        fold_preds = pd.DataFrame({"indice": idx_valid, "y_real": yva.to_numpy()})
    else:
        raw_inner, agg_inner = _run_inner_cv_fase3(Xtr, ytr, outer_key, N_INNER_FASE3)
        if agg_inner.empty:
            warnings.warn(f"Externa {outer_id}: ningún bloque interno evaluable; se omite.", RuntimeWarning)
            fold_metrics = pd.DataFrame([{"particion_externa": outer_id, "modelo": "",
                                         "estado": "no_evaluable", "n_validacion": len(yva),
                                         "spikes_validacion": int(yva.sum())}])
            fold_preds = pd.DataFrame({"indice": idx_valid, "y_real": yva.to_numpy()})
        else:
            evaluable_outer += 1
            fold_metric_rows = []
            fold_preds = pd.DataFrame({"indice": idx_valid, "y_real": yva.to_numpy()})
            trivial = _measure_threshold_fase3(yva, None, None, "Referencia trivial", "{}", outer_id, True)
            fold_metric_rows.append({**trivial, "particion_externa": outer_id,
                                     "estado": "referencia_trivial"})
            for model_name in GRILLAS_FASE3:
                selected = _choose_fase3(agg_inner, model_name)
                if selected is None:
                    warnings.warn(f"Externa {outer_id}: sin candidato elegible para {model_name}.", RuntimeWarning)
                    continue
                params = json.loads(selected["parametros"])
                try:
                    fitted, scores = _fit_score_fase3(model_name, params, Xtr, ytr, Xva, N_CAL_FASE3)
                except Exception as exc:
                    warnings.warn(f"Externa {outer_id}, {model_name}: ajuste no disponible: {exc}", RuntimeWarning)
                    continue
                threshold = float(selected["umbral"])
                met = _measure_threshold_fase3(yva, scores, threshold, model_name,
                                               selected["parametros"], outer_id)
                met.update({"particion_externa": outer_id, "estado": "evaluable",
                            "umbral_seleccionado": threshold,
                            "estado_tolerancia_interna": selected["estado_tolerancia"]})
                fold_metric_rows.append(met)
                slug = model_name.lower().replace(" ", "_").replace("ó", "o").replace("í", "i").replace(" ", "_")
                fold_preds[f"{slug}_score"] = np.asarray(scores)
                fold_preds[f"{slug}_alerta"] = (np.asarray(scores) >= threshold).astype(int)
                model_path = FASE3_DIR / f"modelo_{outer_key}_{slug}.joblib"
                joblib.dump({"estimator": fitted, "model_name": model_name,
                             "params": params, "threshold": threshold,
                             "config_hash": CONFIG_HASH_FASE3}, model_path)
            fold_metrics = pd.DataFrame(fold_metric_rows)
    _atomic_csv_fase3(fold_metrics, metrics_path)
    _atomic_csv_fase3(fold_preds, preds_path)
    meta_path.write_text(json.dumps({"config_hash": CONFIG_HASH_FASE3,
                                     "input_hash": outer_hash}, indent=2), encoding="utf-8")
    external_rows.extend(fold_metrics.to_dict("records"))
    prediction_frames.append(fold_preds)

if evaluable_outer == 0:
    raise RuntimeError("Ninguna partición externa fue evaluable; no se reporta rendimiento externo.")
metricas_externas_fase3 = pd.DataFrame(external_rows)
predicciones_externas_fase3 = pd.concat(prediction_frames, ignore_index=True)
evaluable = metricas_externas_fase3[metricas_externas_fase3.estado == "evaluable"].copy()
if evaluable.empty:
    raise RuntimeError("No quedaron métricas externas de candidatos evaluables.")
resumen_externo_fase3 = evaluable.groupby("modelo").agg(
    f2_mean=("f2", "mean"), f2_std=("f2", "std"),
    recall_mean=("recall", "mean"), recall_std=("recall", "std"),
    precision_mean=("precision", "mean"), precision_std=("precision", "std"),
    fpr_mean=("fpr", "mean"), fpr_std=("fpr", "std"),
    alert_rate_mean=("alert_rate", "mean"), alert_rate_std=("alert_rate", "std"),
    falsas_alarmas_mean=("fp", "mean"), falsos_negativos_mean=("fn", "mean"),
    particiones=("particion_externa", "nunique"),
).reset_index()
trivial_by_fold = metricas_externas_fase3[metricas_externas_fase3.estado == "referencia_trivial"].set_index("particion_externa")
comparacion_trivial = []
for model_name, group in evaluable.groupby("modelo"):
    paired = group.set_index("particion_externa").join(
        trivial_by_fold[["f2"]].rename(columns={"f2": "f2_trivial"}), how="inner"
    )
    delta = paired.f2 - paired.f2_trivial
    improves = bool(delta.mean() > 0 and (delta > 0).sum() > len(delta) / 2)
    comparacion_trivial.append({"modelo": model_name, "f2_diferencia_media_vs_trivial": delta.mean(),
                                "particiones_delta_positiva": int((delta > 0).sum()),
                                "particiones_comparables": len(delta),
                                "mejora_descriptiva_clara": improves})
comparacion_referencia_trivial_fase3 = pd.DataFrame(comparacion_trivial)
resumen_externo_fase3 = resumen_externo_fase3.merge(comparacion_referencia_trivial_fase3, on="modelo", how="left")
metricas_externas_fase3.to_csv(FASE3_DIR / "metricas_por_particion_externa.csv", index=False)
resumen_externo_fase3.to_csv(FASE3_DIR / "resumen_externo_fase3.csv", index=False)
predicciones_externas_fase3.to_csv(FASE3_DIR / "predicciones_externas_fase3.csv", index=False)
print(resumen_externo_fase3.to_string(index=False))
if not comparacion_referencia_trivial_fase3.mejora_descriptiva_clara.any():
    print("RESULTADO DESCRIPTIVO: ningún modelo mejora la referencia trivial en F2 externo.")
else:
    print("Al menos un modelo supera descriptivamente la referencia trivial según el criterio predefinido; no es una prueba de significancia.")
if seleccion_global_fase3["estado_tolerancia"] == "no cumple la tolerancia":
    print("ADVERTENCIA: el candidato global se seleccionó por F2 sin cumplir la tolerancia; se conserva la advertencia.")


In [ ]:
# C23T — Cronometraje opcional de un fold en MODO_RAPIDO
from pathlib import Path
from time import perf_counter
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

MEDIR_UN_FOLD_RAPIDO = False
if MEDIR_UN_FOLD_RAPIDO:
    if not MODO_RAPIDO:
        raise RuntimeError("Activa MODO_RAPIDO=True antes de medir un fold diagnóstico.")
    X_t, y_t = X_train_fase2, y_train_fase2.astype(int)
    split = next((pair for pair in TimeSeriesSplit(n_splits=N_INNER_FASE3).split(X_t)
                  if int(y_t.iloc[pair[1]].sum()) >= MIN_SPIKES_BLOQUE
                  and y_t.iloc[pair[0]].nunique() == 2
                  and y_t.iloc[pair[1]].nunique() == 2), None)
    if split is None:
        raise RuntimeError("No hay fold rápido con suficientes spikes y ambas clases.")
    itr, iva = split
    t0 = perf_counter()
    fits_ok, fits_failed = 0, 0
    for model_name, configs in GRILLAS_FASE3.items():
        for params in configs:
            try:
                _fit_score_fase3(model_name, params, X_t.iloc[itr], y_t.iloc[itr],
                                 X_t.iloc[iva], N_CAL_FASE3)
                fits_ok += 1
            except Exception as exc:
                fits_failed += 1
                print(f"No medido para {model_name} {params}: {exc}")
    elapsed = perf_counter() - t0
    row = {"run_id": RUN_ID, "modo": "rapido", "segundos_fold_medidos": elapsed,
           "configuraciones_ok": fits_ok, "configuraciones_fallidas": fits_failed,
           "nota": "Un fold medido; cualquier extrapolacion es aproximada, no un tiempo observado."}
    pd.DataFrame([row]).to_csv(FASE3_DIR / "tiempo_fold_fase3.csv", index=False)
    print(f"Tiempo observado de un fold rápido: {elapsed:.1f} s; tiempo total completo: no medido.")
else:
    print("Cronometraje opcional desactivado. Tiempos de Colab: no medidos.")


In [ ]:
# C24 - Artefacto de desarrollo, SHA-256 y prueba de recarga
from pathlib import Path
import hashlib,json,joblib,numpy as np,pandas as pd
for n in ("seleccion_global_fase3","X_train_fase2","y_train_fase2","_fit_score_fase3","FASE3_DIR","RUN_ID","comparacion_referencia_trivial_fase3"):
    if n not in globals(): raise RuntimeError(f"Falta {n}; ejecuta C22 y C23T.")
sel=dict(seleccion_global_fase3); model=str(sel["modelo"]); params=json.loads(str(sel["parametros"])); threshold=float(sel["umbral"])
trivial_match=comparacion_referencia_trivial_fase3.loc[comparacion_referencia_trivial_fase3.modelo.eq(model)]
selected_beats_trivial=bool(len(trivial_match) and trivial_match.iloc[0]["mejora_descriptiva_clara"])
any_model_beats_trivial=bool(comparacion_referencia_trivial_fase3["mejora_descriptiva_clara"].any())
est,control=_fit_score_fase3(model,params,X_train_fase2,y_train_fase2,X_train_fase2.iloc[[0]],N_CAL_FASE3)
artifact={"estimador":est,"modelo":model,"parametros":params,"umbral":threshold,
 "estado_tolerancia":str(sel["estado_tolerancia"]),
 "cumple_tolerancia":sel["estado_tolerancia"]=="cumple_tolerancia_en_mayoria_interna",
 "modelo_seleccionado_mejora_referencia_trivial":selected_beats_trivial,
 "alguno_mejora_referencia_trivial":any_model_beats_trivial,
 "run_id":RUN_ID,"config_sha256":CONFIG_HASH_FASE3,"features":list(X_train_fase2.columns)}
FASE3_DIR=Path(FASE3_DIR); FASE3_DIR.mkdir(parents=True,exist_ok=True)
MODELO_FINAL_PATH=FASE3_DIR/"seleccion_final_desarrollo.joblib"; joblib.dump(artifact,MODELO_FINAL_PATH)
MODELO_FINAL_SHA256=hashlib.sha256(MODELO_FINAL_PATH.read_bytes()).hexdigest()
def _score_bundle_fase3(bundle,X):
    est=bundle["estimador"]
    if isinstance(est,dict) and {"base","calibrator"}.issubset(est):
        z=np.asarray(est["base"].decision_function(X),dtype=float).reshape(-1,1)
        return est["calibrator"].predict_proba(z)[:,1]
    return est.predict_proba(X)[:,1]
loaded=joblib.load(MODELO_FINAL_PATH)
if not np.allclose(control,_score_bundle_fase3(loaded,X_train_fase2.iloc[[0]]),rtol=0,atol=1e-12):
    raise AssertionError("El score de control se modifico al recargar.")
pd.DataFrame([{"run_id":RUN_ID,"modelo":model,"parametros":json.dumps(params,sort_keys=True),
 "umbral":threshold,"estado_tolerancia":artifact["estado_tolerancia"],
 "cumple_tolerancia":artifact["cumple_tolerancia"],
 "modelo_seleccionado_mejora_referencia_trivial":selected_beats_trivial,
 "alguno_mejora_referencia_trivial":any_model_beats_trivial,
 "alert_rate_validacion_interna":sel["alert_rate"],
 "joblib_path":str(MODELO_FINAL_PATH),"joblib_sha256":MODELO_FINAL_SHA256,
 "config_sha256":CONFIG_HASH_FASE3}]).to_csv(FASE3_DIR/"seleccion_final_desarrollo.csv",index=False)
print(f"Modelo={model}; umbral={threshold}; tolerancia={artifact['estado_tolerancia']}; SHA-256={MODELO_FINAL_SHA256}")
print("Score de control coincide tras recarga.")


In [ ]:
# C25 - Articulo vs adaptacion: BIC en muestra com\u00fan (hold_back=11)
from pathlib import Path
import numpy as np,pandas as pd
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox,het_arch
if "train_retornos" not in globals() or "FASE3_DIR" not in globals(): raise RuntimeError("Ejecuta C09 y C22.")
y=train_retornos.dropna().astype(float)*100
specs={"Articulo AR(11)-EGARCH(1,1)-GED":(11,1,1,1,"ged"),
       "Adaptacion AR(3)-EGARCH(2,1)-t":(3,2,1,1,"t")}
fits={}; rows=[]
for name,(lag,p,o,q,dist) in specs.items():
    fit=arch_model(y,mean="AR",lags=lag,vol="EGARCH",p=p,o=o,q=q,dist=dist,hold_back=11).fit(disp="off",show_warning=False)
    if fit.convergence_flag: raise RuntimeError(f"{name} no convergio: {fit.convergence_flag}")
    fits[name]=fit
    rows.append({"especificacion":name,"bic":float(fit.bic),"muestra":"muestra com\u00fan (hold_back=11)",
                 "nobs":int(fit.nobs),"nota":"No comparable con BIC de C09: hold-back/muestra efectiva distinta."})
common=None
for fit in fits.values():
    idx=pd.DatetimeIndex(fit.resid.index[fit.resid.notna()])
    common=idx if common is None else common.intersection(idx)
if common is None or len(common)==0: raise RuntimeError("No hay fechas comunes.")
labels={}; diags=[]
for name,fit in fits.items():
    resid=fit.resid.reindex(common); sigma=fit.conditional_volatility.reindex(common)
    labels[name]=(resid.abs()>2*sigma).astype(int)
    std=pd.Series(np.asarray(fit.std_resid),index=fit.resid.index).reindex(common).dropna()
    lb=acorr_ljungbox(std,lags=[10],return_df=True)
    lm,pv,_,_=het_arch(std,nlags=min(10,len(std)-1))
    diags.append({"especificacion":name,"muestra":"muestra com\u00fan (hold_back=11)","n_fechas":len(common),
      "bic":float(fit.bic),"ljung_box_lag10_p":float(lb.lb_pvalue.iloc[0]),"arch_lm":float(lm),
      "arch_lm_p":float(pv),"spikes":int(labels[name].sum()),"prevalencia":labels[name].mean()})
agreement=float((labels[list(labels)[0]]==labels[list(labels)[1]]).mean())
bic=pd.DataFrame(rows); bic["acuerdo_etiquetas"]=agreement
diag=pd.DataFrame(diags); diag["acuerdo_etiquetas"]=agreement
bic.to_csv(Path(FASE3_DIR)/"comparacion_bic_articulo.csv",index=False)
diag.to_csv(Path(FASE3_DIR)/"diagnosticos_articulo.csv",index=False)
print("BIC de muestra com\u00fan (hold_back=11); no comparar con C09.")
print(bic.to_string(index=False)); print(diag.to_string(index=False))
print("Etiquetas in-sample; no constituye comparacion de clasificadores.")


In [ ]:
# C26 - Importancia por permutacion sobre validacion externa; SHAP opcional
from pathlib import Path
import unicodedata,warnings,joblib,numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
if "metricas_externas_fase3" not in globals() or "cv_externo_fase3" not in globals(): raise RuntimeError("Ejecuta C23.")
if "_score_bundle_fase3" not in globals(): raise RuntimeError("Ejecuta C24 para definir el helper de puntuacion.")

def _score_outer_bundle_fase3(bundle, X):
    """Normaliza el nombre de la clave usado por los artefactos de C23/C24."""
    if not isinstance(bundle, dict):
        raise TypeError("El artefacto externo debe ser un diccionario.")
    estimator = bundle.get("estimador", bundle.get("estimator"))
    if estimator is None:
        raise KeyError("El artefacto no contiene 'estimador' ni 'estimator'.")
    normalized = dict(bundle)
    normalized["estimador"] = estimator
    return _score_bundle_fase3(normalized, X)

def _outer_bundle_threshold_fase3(bundle):
    if "threshold" in bundle:
        return float(bundle["threshold"])
    if "umbral" in bundle:
        return float(bundle["umbral"])
    raise KeyError("El artefacto externo no contiene 'threshold' ni 'umbral'.")
model=str(seleccion_global_fase3["modelo"]); rows=[]; error_rows=[]
if model=="Mayor\u00eda":
    warnings.warn("Mayor\u00eda se excluye de importancia por permutacion: no tiene predictores.",RuntimeWarning)
else:
    eligible=metricas_externas_fase3[(metricas_externas_fase3.estado=="evaluable")&(metricas_externas_fase3.modelo==model)]
    if eligible.empty: raise RuntimeError("No hay folds externos para el modelo seleccionado.")
    slug=unicodedata.normalize("NFKD",model).encode("ascii","ignore").decode().lower().replace(" ","_")
    rng=np.random.default_rng(42); repeats=2 if MODO_RAPIDO else 10
    for row in eligible.itertuples(index=False):
        fold=int(row.particion_externa); _,iv=cv_externo_fase3[fold-1]
        Xv=X_train_fase2.iloc[iv]; yv=y_train_fase2.iloc[iv].astype(int)
        if int(yv.sum())<MIN_SPIKES_BLOQUE: continue
        mp=Path(FASE3_DIR)/f"modelo_outer_{fold}_{slug}.joblib"
        if not mp.is_file(): raise FileNotFoundError(mp)
        bundle=joblib.load(mp); base=_score_outer_bundle_fase3(bundle,Xv); ap=average_precision_score(yv,base)
        pred=base>=_outer_bundle_threshold_fase3(bundle)
        error_rows.extend([{"particion":fold,"fecha":str(train_fase2.iloc[iv[j]]["date"]),
           "y_real":int(yv.iloc[j]),"score":float(base[j]),"alerta":int(pred[j]),
           "tipo_error":"falsa_alarma" if not yv.iloc[j] and pred[j] else
                        "spike_omitido" if yv.iloc[j] and not pred[j] else "acierto"} for j in range(len(iv))])
        for col in Xv:
            drops=[]
            for _ in range(repeats):
                xp=Xv.copy(); xp[col]=xp[col].to_numpy()[rng.permutation(len(xp))]
                drops.append(ap-average_precision_score(yv,_score_outer_bundle_fase3(bundle,xp)))
            rows.append({"modelo":model,"particion_externa":fold,"variable":col,
                         "caida_ap_media":float(np.mean(drops)),"caida_ap_de":float(np.std(drops,ddof=1)) if repeats>1 else 0})
    tabla_importancia_fase3=pd.DataFrame(rows)
    if tabla_importancia_fase3.empty: raise RuntimeError("No se pudo calcular permutacion externa.")
    tabla_importancia_fase3.to_csv(Path(FASE3_DIR)/"importancia_permutacion_externa.csv",index=False)
    pd.DataFrame(error_rows).to_csv(Path(FASE3_DIR)/"casos_error_validacion_externa.csv",index=False)
    imp=tabla_importancia_fase3.groupby("variable").caida_ap_media.mean().sort_values()
    imp.plot.barh(figsize=(9,max(4,len(imp)*.35)),title="Permutacion en validacion externa")
    plt.tight_layout(); plt.savefig(Path(FASE3_DIR)/"importancia_permutacion_externa.png",dpi=180); plt.show()
    print("Nunca se uso el test. Predictores correlacionados pueden repartirse importancia; no es causalidad.")
SHAP_LOCAL_FASE3=False
if SHAP_LOCAL_FASE3 and model!="Mayor\u00eda":
    try:
        import shap
        fold=int(eligible.iloc[-1]["particion_externa"]); it,iv=cv_externo_fase3[fold-1]
        small=X_train_fase2.iloc[iv].head(10); background=X_train_fase2.iloc[it].head(20)
        slug=unicodedata.normalize("NFKD",model).encode("ascii","ignore").decode().lower().replace(" ","_")
        bundle_shap=joblib.load(Path(FASE3_DIR)/f"modelo_outer_{fold}_{slug}.joblib")
        def predict_shap(values):
            frame=pd.DataFrame(values,columns=X_train_fase2.columns)
            return _score_outer_bundle_fase3(bundle_shap,frame)
        explainer=shap.KernelExplainer(predict_shap,background.to_numpy())
        values=explainer.shap_values(small.to_numpy(),nsamples=50)
        np.save(Path(FASE3_DIR)/"shap_local_validacion_externa.npy",np.asarray(values))
        print("SHAP local calculado en hasta 10 filas de validacion externa; no test.")
    except Exception as e: warnings.warn(f"SHAP opcional fallo: {type(e).__name__}: {e}",RuntimeWarning)
else: print("SHAP opcional desactivado; metodo principal: permutacion externa.")


## C27 - Preparacion de la app de reproduccion historica
Copia el artefacto de C24 y prepara entradas unicamente del tramo de desarrollo. No descarga datos ni usa el test.

In [ ]:
# C27 - Preparar modelo y datos historicos locales para Streamlit
from pathlib import Path
import hashlib,json,os,shutil,pandas as pd
if "MODELO_FINAL_PATH" not in globals() or "train_fase2" not in globals(): raise RuntimeError("Ejecuta C24 y C17.")
if hashlib.sha256(Path(MODELO_FINAL_PATH).read_bytes()).hexdigest()!=MODELO_FINAL_SHA256: raise RuntimeError("Hash del modelo se modifico.")
os.chdir(PROJECT_DIR)  # C28 usa %%writefile relativo; garantiza que escriba dentro del repositorio.
APP_DIR=Path(PROJECT_DIR)/"app"; (APP_DIR/"data").mkdir(parents=True,exist_ok=True)
shutil.copy2(MODELO_FINAL_PATH,APP_DIR/"model.joblib")
if hashlib.sha256((APP_DIR/"model.joblib").read_bytes()).hexdigest()!=MODELO_FINAL_SHA256: raise AssertionError("Hash de copia no coincide.")
history=X_train_fase2.copy(); history.insert(0,"fecha",pd.to_datetime(train_fase2["date"]).to_numpy())
history.to_csv(APP_DIR/"data"/"reproduccion_historica.csv",index=False)
(APP_DIR/"app_metadata.json").write_text(json.dumps({"run_id":RUN_ID,"model_sha256":MODELO_FINAL_SHA256},indent=2),encoding="utf-8")
print("Artefacto y datos de desarrollo preparados para reproduccion historica.")


In [ ]:
%%writefile app/app.py
from pathlib import Path
import hashlib,json,joblib,numpy as np,pandas as pd,streamlit as st
D=Path(__file__).resolve().parent; M=D/"model.joblib"; X=D/"data"/"reproduccion_historica.csv"
st.set_page_config(page_title="TRM historica")
st.title("Reproduccion historica de alertas de TRM")
st.warning("Reproduce scores historicos locales; no es prediccion en vivo.")
if not M.is_file() or not X.is_file(): st.error("Ejecuta C24 y C27 en el notebook."); st.stop()
b=joblib.load(M); meta=json.loads((D/"app_metadata.json").read_text(encoding="utf-8"))
if hashlib.sha256(M.read_bytes()).hexdigest()!=meta["model_sha256"]: st.error("Hash del modelo no coincide."); st.stop()
df=pd.read_csv(X,parse_dates=["fecha"]); features=b["features"]
st.caption(f'{b["modelo"]} | umbral={b["umbral"]:.5f} | run_id={b["run_id"]}')
if not b["cumple_tolerancia"]: st.error("El modelo NO cumple la tolerancia predefinida.")
if not b.get("alguno_mejora_referencia_trivial",False): st.warning("Ningun modelo mejora claramente alertar siempre en F2 externo.")
day=st.selectbox("Fecha historica",df.fecha.dt.strftime("%Y-%m-%d"))
row=df.loc[df.fecha.dt.strftime("%Y-%m-%d").eq(day),features].iloc[[0]]
est=b["estimador"]
if isinstance(est,dict) and {"base","calibrator"}.issubset(est):
    z=np.asarray(est["base"].decision_function(row)).reshape(-1,1); score=float(est["calibrator"].predict_proba(z)[0,1])
else: score=float(est.predict_proba(row)[0,1])
st.metric("Score",f"{score:.4f}"); st.metric("Umbral",f'{b["umbral"]:.4f}')
st.success("ALERTA" if score>=b["umbral"] else "Sin alerta")


In [ ]:
# C29 - Evaluacion unica del test congelado; no interviene en seleccion
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,joblib,numpy as np,pandas as pd
from sklearn.metrics import accuracy_score,average_precision_score,confusion_matrix,f1_score,precision_score,recall_score,roc_auc_score
if "X_test_fase2" not in globals() or "y_test_fase2" not in globals(): raise RuntimeError("Ejecuta C17.")
for n in ("MODELO_FINAL_PATH","MODELO_FINAL_SHA256","RUN_ID","FASE3_DIR","_score_bundle_fase3"):
    if n not in globals(): raise RuntimeError(f"Falta {n}; ejecuta C24.")
reg=Path(FASE3_DIR)/"seleccion_final_desarrollo.csv"
if not reg.is_file(): raise FileNotFoundError("C29 exige seleccion_final_desarrollo.csv.")
r=pd.read_csv(reg)
if len(r)!=1 or str(r.iloc[0].run_id)!=str(RUN_ID): raise RuntimeError("Run ID de seleccion no coincide.")
digest=hashlib.sha256(Path(MODELO_FINAL_PATH).read_bytes()).hexdigest()
if digest!=str(r.iloc[0].joblib_sha256) or digest!=MODELO_FINAL_SHA256: raise RuntimeError("SHA-256 no coincide.")
marker=Path(FASE3_DIR)/f"C29_test_evaluation_{RUN_ID}.json"
if marker.exists(): raise RuntimeError("C29 ya se inicio para este run_id; no se reutiliza el test.")
state={"run_id":RUN_ID,"status":"started","test_ya_reportado_en_fase2":True,"usado_para_seleccion":False,"modelo_sha256":digest}
marker.write_text(json.dumps(state,indent=2),encoding="utf-8")
try:
    if len(y_test_fase2)!=684 or int(y_test_fase2.sum())!=33:
        print(f"WARNING: test={len(y_test_fase2)} rows, {int(y_test_fase2.sum())} spikes; frozen reference=684/33.")
    a=joblib.load(MODELO_FINAL_PATH); yt=np.asarray(y_test_fase2,dtype=int)
    score=_score_bundle_fase3(a,X_test_fase2); threshold=float(a["umbral"]); yp=(score>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(yt,yp,labels=[0,1]).ravel()
    pr=precision_score(yt,yp,zero_division=0); rc=recall_score(yt,yp,zero_division=0)
    result={"run_id":RUN_ID,"modelo":a["modelo"],"parametros":json.dumps(a["parametros"],sort_keys=True),
      "umbral":threshold,"estado_tolerancia":a["estado_tolerancia"],"alert_rate":yp.mean(),
      "n_test":len(yt),"spikes_test":int(yt.sum()),"accuracy":accuracy_score(yt,yp),"recall":rc,
      "precision":pr,"f1":f1_score(yt,yp,zero_division=0),"f2":5*pr*rc/max(4*pr+rc,1e-15),
      "roc_auc":roc_auc_score(yt,score) if len(np.unique(yt))==2 else np.nan,
      "average_precision":average_precision_score(yt,score),"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
      "matriz_confusion":json.dumps([[int(tn),int(fp)],[int(fn),int(tp)]]),"modelo_sha256":digest,
      "test_ya_reportado_en_fase2":True,"usado_para_seleccion":False}
    pd.DataFrame([result]).to_csv(Path(FASE3_DIR)/"evaluacion_test_congelado_unica.csv",index=False)
    pd.DataFrame({"fecha":pd.to_datetime(test_fase2.date),"y_real":yt,"score":score,"umbral":threshold,"alerta":yp}).to_csv(Path(FASE3_DIR)/"predicciones_test_congelado_unica.csv",index=False)
    state.update({"status":"completed","completed_utc":datetime.now(timezone.utc).isoformat()})
    marker.write_text(json.dumps(state,indent=2),encoding="utf-8")
except Exception as e:
    state.update({"status":"failed_no_retry","error":f"{type(e).__name__}: {e}"})
    marker.write_text(json.dumps(state,indent=2),encoding="utf-8")
    raise
print("Test congelado evaluado una vez; ya reportado en Fase 2 y excluido de la seleccion.")
print(pd.DataFrame([result]).to_string(index=False))


In [ ]:
# C30 - Guion de sustentacion y demostracion Streamlit
from pathlib import Path
import os,socket,subprocess,sys,time
if "PROJECT_DIR" not in globals() or "FASE3_DIR" not in globals(): raise RuntimeError("Ejecuta C02 y C22.")
APP_DIR=Path(PROJECT_DIR)/"app"; APP=APP_DIR/"app.py"
for p in (APP,APP_DIR/"model.joblib",APP_DIR/"data"/"reproduccion_historica.csv"):
    if not p.is_file(): raise FileNotFoundError(f"Falta {p}; ejecuta C27-C28.")
guide=Path(PROJECT_DIR)/"docs"/"guion_sustentacion_fase3.md"
if not guide.is_file(): raise FileNotFoundError(guide)
print("Guion:",guide)
try:
    import importlib.metadata as md; md.version("streamlit")
except Exception: print("Streamlit no esta instalado; instala el pin indicado en requirements-colab.txt.")
else:
    port=8501
    with socket.socket() as s: listening=s.connect_ex(("127.0.0.1",port))==0
    if not listening:
        log=open(Path(FASE3_DIR)/"streamlit_app.log","w",encoding="utf-8")
        subprocess.Popen([sys.executable,"-m","streamlit","run",str(APP),"--server.port",str(port),"--server.headless","true"],cwd=PROJECT_DIR,stdout=log,stderr=subprocess.STDOUT)
        for _ in range(30):
            time.sleep(1)
            with socket.socket() as s:
                if s.connect_ex(("127.0.0.1",port))==0: break
        else: raise RuntimeError(f"Streamlit no abrio {port}; revisa el log.")
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
        from google.colab import output
        output.serve_kernel_port_as_iframe(port,height=720)
    else: print(f"App local: http://localhost:{port}")
